<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</center>
</figure>

# **Universidad de Buenos Aires**
## **Facultad de Ciencias Económicas**

### **Taller de Programación para el Análisis de Datos**

# 🛒 Estadística Descriptiva Aplicada
## Dataset: Online Retail UK — UCI Machine Learning Repository

---

### Contexto de negocio

Trabajamos con datos reales de una **tienda mayorista online del Reino Unido** (2010–2011).
La empresa vende artículos de regalo a minoristas de todo el mundo.

| Variable | Descripción |
|---|---|
| `InvoiceNo` | Número de factura (si empieza con 'C' → devolución) |
| `StockCode` | Código de producto |
| `Description` | Nombre del producto |
| `Quantity` | Unidades por transacción (negativo = devolución) |
| `InvoiceDate` | Fecha y hora de la transacción |
| `UnitPrice` | Precio unitario en libras esterlinas (£) |
| `CustomerID` | Identificador del cliente |
| `Country` | País del cliente |

**Fuente:** [UCI Machine Learning Repository — Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail)

---
### Objetivos de la clase
- Aplicar medidas de posición, dispersión y asimetría sobre datos reales
- Ver por qué la **media no es suficiente** en distribuciones asimétricas
- Practicar limpieza de datos antes del análisis
- Comparar distribuciones entre grupos (países, meses)

---
## 0. Carga del dataset

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [2]:
# El dataset está disponible en el repositorio UCI.
# Opción A: carga directa desde URL (requiere conexión a internet)
# Opción B: descargá el archivo desde https://archive.ics.uci.edu/dataset/352/online+retail
#           y descomprimilo en la misma carpeta que esta notebook.

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

try:
    df_raw = pd.read_excel(URL, engine='openpyxl')
    print("Dataset cargado desde URL")
except Exception:
    try:
        df_raw = pd.read_excel("Online Retail.xlsx", engine='openpyxl')
        print("Dataset cargado desde archivo local")
    except FileNotFoundError:
        print("No se pudo cargar el dataset.")
        print("Descargalo desde: https://archive.ics.uci.edu/dataset/352/online+retail")
        print("y colocá 'Online Retail.xlsx' en la misma carpeta que esta notebook.")
        df_raw = None

if df_raw is not None:
    print(f"\nFilas: {len(df_raw):,}")
    print(f"Columnas: {df_raw.columns.tolist()}")
    df_raw.head()

Dataset cargado desde URL

Filas: 541,909
Columnas: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


---
## 1. Exploración inicial y limpieza

Los datos reales casi siempre tienen problemas. Antes de calcular cualquier estadística, entendemos qué tenemos.

In [3]:
print("── Tipos de datos ──")
print(df_raw.dtypes)
print(f"\n── Valores nulos por columna ──")
nulos = df_raw.isnull().sum()
print(nulos[nulos > 0])
print(f"\nTotal de filas originales: {len(df_raw):,}")

── Tipos de datos ──
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

── Valores nulos por columna ──
Description      1454
CustomerID     135080
dtype: int64

Total de filas originales: 541,909


In [4]:
# ── Decisiones de limpieza ────────────────────────────────────────
#
# 1. Eliminar filas sin CustomerID (no identificables)
# 2. Separar devoluciones (InvoiceNo que empieza con 'C') para análisis aparte
# 3. Eliminar precios = 0 (productos gratuitos / errores de carga)
# 4. Eliminar cantidades <= 0 en el dataset de ventas

# Separamos devoluciones ANTES de limpiar
devoluciones = df_raw[df_raw['InvoiceNo'].astype(str).str.startswith('C')].copy()
ventas_raw   = df_raw[~df_raw['InvoiceNo'].astype(str).str.startswith('C')].copy()

# Aplicamos limpieza sobre ventas
df = ventas_raw.copy()
df = df.dropna(subset=['CustomerID'])
df = df[df['UnitPrice'] > 0]
df = df[df['Quantity'] > 0]

# Feature engineering: Revenue = ingresos por línea de factura
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Extraemos componentes de fecha
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Mes']         = df['InvoiceDate'].dt.month
df['DiaSemana']   = df['InvoiceDate'].dt.day_name()

print(f"Filas originales : {len(df_raw):,}")
print(f"Devoluciones     : {len(devoluciones):,}")
print(f"Ventas limpias   : {len(df):,}")
print(f"\nPeriodo: {df['InvoiceDate'].min().date()}  →  {df['InvoiceDate'].max().date()}")
print(f"Países  : {df['Country'].nunique()}")
print(f"Clientes: {df['CustomerID'].nunique():,}")
print(f"Productos: {df['StockCode'].nunique():,}")
df.head()

Filas originales : 541,909
Devoluciones     : 9,288
Ventas limpias   : 397,884

Periodo: 2010-12-01  →  2011-12-09
Países  : 37
Clientes: 4,338
Productos: 3,665


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Mes,DiaSemana
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,12,Wednesday
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,12,Wednesday
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,12,Wednesday
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,12,Wednesday
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,12,Wednesday


---
## 2. Análisis descriptivo — Precio unitario (`UnitPrice`)

Empezamos por el precio. En datos de ventas, la distribución de precios suele ser **muy asimétrica**: la mayoría de los productos son baratos, pero algunos tienen precios muy altos.

In [5]:
precios = df['UnitPrice']

print("═" * 50)
print("  PRECIO UNITARIO (£)")
print("─" * 50)
print(f"  n                  : {len(precios):,}")
print(f"── Posición ──")
print(f"  Media              : £{precios.mean():.4f}")
print(f"  Mediana            : £{precios.median():.4f}")
print(f"  Moda               : £{precios.mode()[0]:.4f}")
print(f"  P10 / P25 / P75 / P90: £{precios.quantile(0.10):.2f} / £{precios.quantile(0.25):.2f} / £{precios.quantile(0.75):.2f} / £{precios.quantile(0.90):.2f}")
print(f"── Dispersión ──")
print(f"  Mín / Máx          : £{precios.min():.2f} / £{precios.max():.2f}")
print(f"  Rango              : £{precios.max() - precios.min():.2f}")
print(f"  RIQ                : £{precios.quantile(0.75) - precios.quantile(0.25):.4f}")
print(f"  Desvío estándar    : £{precios.std():.4f}")
print(f"  CV                 : {precios.std()/precios.mean()*100:.1f}%")
print(f"── Asimetría ──")
print(f"  Fisher G1          : {precios.skew():.4f}")
print("═" * 50)

print("\n💡 Observación:")
print(f"   La media (£{precios.mean():.2f}) es MUY superior a la mediana (£{precios.median():.2f}).")
print(f"   Esto indica que unos pocos productos de precio muy alto 'tiran' la media hacia arriba.")
print(f"   CV = {precios.std()/precios.mean()*100:.0f}% → dispersión altísima respecto a la media.")

══════════════════════════════════════════════════
  PRECIO UNITARIO (£)
──────────────────────────────────────────────────
  n                  : 397,884
── Posición ──
  Media              : £3.1165
  Mediana            : £1.9500
  Moda               : £1.2500
  P10 / P25 / P75 / P90: £0.55 / £1.25 / £3.75 / £6.35
── Dispersión ──
  Mín / Máx          : £0.00 / £8142.75
  Rango              : £8142.75
  RIQ                : £2.5000
  Desvío estándar    : £22.0979
  CV                 : 709.1%
── Asimetría ──
  Fisher G1          : 204.0327
══════════════════════════════════════════════════

💡 Observación:
   La media (£3.12) es MUY superior a la mediana (£1.95).
   Esto indica que unos pocos productos de precio muy alto 'tiran' la media hacia arriba.
   CV = 709% → dispersión altísima respecto a la media.


In [6]:
# ¿Cuánto representa el P90 vs el máximo?
p90 = precios.quantile(0.90)
p99 = precios.quantile(0.99)

print(f"El 90% de los productos tiene precio ≤ £{p90:.2f}")
print(f"El 99% de los productos tiene precio ≤ £{p99:.2f}")
print(f"El precio máximo es £{precios.max():.2f}")
print(f"\nProductos con precio > £{p99:.0f} (top 1%):")
print(df[df['UnitPrice'] > p99][['Description', 'UnitPrice', 'Country']]
      .sort_values('UnitPrice', ascending=False)
      .drop_duplicates('Description')
      .head(10)
      .to_string(index=False))

El 90% de los productos tiene precio ≤ £6.35
El 99% de los productos tiene precio ≤ £14.95
El precio máximo es £8142.75

Productos con precio > £15 (top 1%):
                       Description  UnitPrice        Country
                           POSTAGE    8142.75 United Kingdom
                            Manual    4161.06         France
                    DOTCOM POSTAGE    1599.26 United Kingdom
    PICNIC BASKET WICKER 60 PIECES     649.50 United Kingdom
       VINTAGE RED KITCHEN CABINET     295.00 United Kingdom
      VINTAGE BLUE KITCHEN CABINET     295.00 United Kingdom
     LOVE SEAT ANTIQUE WHITE METAL     195.00 United Kingdom
      REGENCY MIRROR WITH SHUTTERS     165.00 United Kingdom
RUSTIC  SEVENTEEN DRAWER SIDEBOARD     165.00           EIRE
                          CARRIAGE     150.00         France


### ¿Qué pasa si excluimos los outliers?

Comparamos las estadísticas con y sin el top 1% de precios.

In [7]:
precios_sin_outliers = precios[precios <= p99]

print(f"{'Medida':<22} {'Con outliers':>14} {'Sin top 1%':>14} {'Diferencia':>12}")
print("─" * 65)

medidas = [
    ("Media",           precios.mean(),              precios_sin_outliers.mean()),
    ("Mediana",         precios.median(),             precios_sin_outliers.median()),
    ("Desvío estándar", precios.std(),                precios_sin_outliers.std()),
    ("CV (%)",          precios.std()/precios.mean()*100, precios_sin_outliers.std()/precios_sin_outliers.mean()*100),
    ("Asimetría G1",    precios.skew(),               precios_sin_outliers.skew()),
]

for nombre, con, sin in medidas:
    diff_pct = (sin - con) / con * 100
    print(f"{nombre:<22} {con:>14.4f} {sin:>14.4f} {diff_pct:>+11.1f}%")

print("\n💡 La mediana cambia muy poco (-0.X%) → es ROBUSTA ante outliers.")
print("   La media y el desvío se ven mucho más afectados.")

Medida                   Con outliers     Sin top 1%   Diferencia
─────────────────────────────────────────────────────────────────
Media                          3.1165         2.7129       -13.0%
Mediana                        1.9500         1.7900        -8.2%
Desvío estándar               22.0979         2.5355       -88.5%
CV (%)                       709.0635        93.4608       -86.8%
Asimetría G1                 204.0327         1.9346       -99.1%

💡 La mediana cambia muy poco (-0.X%) → es ROBUSTA ante outliers.
   La media y el desvío se ven mucho más afectados.


---
## 3. Análisis descriptivo — Cantidad por transacción (`Quantity`)

In [8]:
cantidad = df['Quantity']

print("═" * 50)
print("  CANTIDAD POR TRANSACCIÓN (unidades)")
print("─" * 50)
print(f"  Media              : {cantidad.mean():.2f} unidades")
print(f"  Mediana            : {cantidad.median():.0f} unidades")
print(f"  Moda               : {cantidad.mode()[0]:.0f} unidades")
print(f"  P25 / P75          : {cantidad.quantile(0.25):.0f} / {cantidad.quantile(0.75):.0f}")
print(f"  RIQ                : {cantidad.quantile(0.75) - cantidad.quantile(0.25):.0f}")
print(f"  Rango              : {cantidad.min()} → {cantidad.max()}")
print(f"  Desvío estándar    : {cantidad.std():.2f}")
print(f"  CV                 : {cantidad.std()/cantidad.mean()*100:.1f}%")
print(f"  Asimetría G1       : {cantidad.skew():.4f}")
print("═" * 50)

# ¿Hay pedidos muy grandes?
p99_q = cantidad.quantile(0.99)
grandes = df[df['Quantity'] >= p99_q][['Description', 'Quantity', 'UnitPrice', 'Country']]
print(f"\nPedidos más grandes (top 1%, Quantity ≥ {p99_q:.0f} unidades):")
print(grandes.sort_values('Quantity', ascending=False).head(8).to_string(index=False))

══════════════════════════════════════════════════
  CANTIDAD POR TRANSACCIÓN (unidades)
──────────────────────────────────────────────────
  Media              : 12.99 unidades
  Mediana            : 6 unidades
  Moda               : 1 unidades
  P25 / P75          : 2 / 12
  RIQ                : 10
  Rango              : 1 → 80995
  Desvío estándar    : 179.33
  CV                 : 1380.7%
  Asimetría G1       : 409.8930
══════════════════════════════════════════════════

Pedidos más grandes (top 1%, Quantity ≥ 120 unidades):
                        Description  Quantity  UnitPrice        Country
        PAPER CRAFT , LITTLE BIRDIE     80995       2.08 United Kingdom
     MEDIUM CERAMIC TOP STORAGE JAR     74215       1.04 United Kingdom
  WORLD WAR 2 GLIDERS ASSTD DESIGNS      4800       0.21 United Kingdom
               SMALL POPCORN HOLDER      4300       0.72 United Kingdom
              EMPIRE DESIGN ROSETTE      3906       0.82 United Kingdom
ESSENTIAL BALM 3.5g TIN IN ENVELO

---
## 4. Análisis descriptivo — Revenue por transacción

El **Revenue** (Quantity × UnitPrice) es la variable de negocio más importante. En e-commerce, la distribución del revenue es casi siempre **extremadamente asimétrica**: pocas transacciones generan la mayor parte de los ingresos.

In [9]:
rev = df['Revenue']

print("═" * 52)
print("  REVENUE POR TRANSACCIÓN (£)")
print("─" * 52)
print(f"  Media              : £{rev.mean():>10.2f}")
print(f"  Mediana            : £{rev.median():>10.2f}")
print(f"  P25 / P75 / P90    : £{rev.quantile(0.25):.2f} / £{rev.quantile(0.75):.2f} / £{rev.quantile(0.90):.2f}")
print(f"  Mín / Máx          : £{rev.min():.2f} / £{rev.max():,.2f}")
print(f"  Desvío estándar    : £{rev.std():>10.2f}")
print(f"  CV                 : {rev.std()/rev.mean()*100:.1f}%")
print(f"  Asimetría G1       : {rev.skew():.4f}")
print("═" * 52)

# Regla de Pareto: ¿el 20% de las transacciones genera el 80% del revenue?
rev_ordenado = rev.sort_values(ascending=False)
rev_total    = rev_ordenado.sum()
n_total      = len(rev_ordenado)

for pct_trans in [0.05, 0.10, 0.20]:
    top_n    = int(n_total * pct_trans)
    top_rev  = rev_ordenado.iloc[:top_n].sum()
    pct_rev  = top_rev / rev_total * 100
    print(f"  Top {pct_trans*100:.0f}% de transacciones → {pct_rev:.1f}% del revenue total")

════════════════════════════════════════════════════
  REVENUE POR TRANSACCIÓN (£)
────────────────────────────────────────────────────
  Media              : £     22.40
  Mediana            : £     11.80
  P25 / P75 / P90    : £4.68 / £19.80 / £35.40
  Mín / Máx          : £0.00 / £168,469.60
  Desvío estándar    : £    309.07
  CV                 : 1380.0%
  Asimetría G1       : 451.4432
════════════════════════════════════════════════════
  Top 5% de transacciones → 43.8% del revenue total
  Top 10% de transacciones → 54.4% del revenue total
  Top 20% de transacciones → 66.5% del revenue total


---
## 5. Análisis por país

Comparamos las estadísticas descriptivas de revenue entre los principales países.

In [10]:
# Top 10 países por revenue total
top_paises = (df.groupby('Country')['Revenue']
                .sum()
                .sort_values(ascending=False)
                .head(10)
                .index.tolist())

df_top = df[df['Country'].isin(top_paises)]

resumen_paises = (df_top.groupby('Country')['Revenue']
                  .agg(
                      transacciones='count',
                      revenue_total='sum',
                      media='mean',
                      mediana='median',
                      desvio='std',
                      asimetria='skew'
                  )
                  .sort_values('revenue_total', ascending=False)
                  .round(2))

resumen_paises['CV (%)'] = (resumen_paises['desvio'] / resumen_paises['media'] * 100).round(1)

print(resumen_paises.to_string())

print("\n💡 Notá que en casi todos los países: media >> mediana → distribución asimétrica positiva.")

                transacciones  revenue_total   media  mediana  desvio  asimetria  CV (%)
Country                                                                                 
United Kingdom         354321     7308391.55   20.63     10.2  326.04     431.79  1580.4
Netherlands              2359      285446.34  121.00     91.8  164.05      13.62   135.6
EIRE                     7236      265545.90   36.70     17.4   83.94      11.15   228.7
Germany                  9040      228867.14   25.32     17.0   35.46       8.44   140.0
France                   8341      209024.05   25.06     16.6   74.36      43.50   296.7
Australia                1182      138521.31  117.19     66.0  159.90       3.43   136.4
Spain                    2484       61577.11   24.79     15.0   70.35      14.15   283.8
Switzerland              1841       56443.95   30.66     17.7   35.82       3.73   116.8
Belgium                  2031       41196.34   20.28     16.6   15.25       3.93    75.2
Sweden               

In [11]:
# Comparación directa entre UK y el segundo país
pais_2 = top_paises[1]

uk   = df[df['Country'] == 'United Kingdom']['Revenue']
otro = df[df['Country'] == pais_2]['Revenue']

print(f"{'Medida':<22} {'United Kingdom':>16} {pais_2:>16}")
print("─" * 56)
for nombre, val_uk, val_otro in [
    ("n transacciones",  f"{len(uk):,}",              f"{len(otro):,}"),
    ("Media (£)",        f"{uk.mean():.2f}",           f"{otro.mean():.2f}"),
    ("Mediana (£)",      f"{uk.median():.2f}",         f"{otro.median():.2f}"),
    ("Desvío (£)",       f"{uk.std():.2f}",            f"{otro.std():.2f}"),
    ("CV (%)",           f"{uk.std()/uk.mean()*100:.1f}%", f"{otro.std()/otro.mean()*100:.1f}%"),
    ("Asimetría G1",     f"{uk.skew():.4f}",           f"{otro.skew():.4f}"),
]:
    print(f"{nombre:<22} {val_uk:>16} {val_otro:>16}")

Medida                   United Kingdom      Netherlands
────────────────────────────────────────────────────────
n transacciones                 354,321            2,359
Media (£)                         20.63           121.00
Mediana (£)                       10.20            91.80
Desvío (£)                       326.04           164.05
CV (%)                          1580.7%           135.6%
Asimetría G1                   431.7927          13.6182


---
## 6. Análisis temporal — Revenue por mes

Usamos estadística descriptiva para detectar **estacionalidad**: si algunos meses tienen distribuciones de revenue distintas.

In [12]:
meses = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
         7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

resumen_mes = (df.groupby('Mes')['Revenue']
               .agg(
                   transacciones='count',
                   revenue_total='sum',
                   media='mean',
                   mediana='median',
                   desvio='std',
                   asimetria='skew'
               )
               .round(2))

resumen_mes.index = resumen_mes.index.map(meses)
resumen_mes['CV (%)'] = (resumen_mes['desvio'] / resumen_mes['media'] * 100).round(1)

print(resumen_mes.to_string())

mes_max = resumen_mes['revenue_total'].idxmax()
mes_min = resumen_mes['revenue_total'].idxmin()
print(f"\nMes con mayor revenue total: {mes_max}")
print(f"Mes con menor revenue total: {mes_min}")

print("\n💡 ¿Hay un pico de ventas hacia fin de año? (típico en artículos de regalo)")

     transacciones  revenue_total  media  mediana  desvio  asimetria  CV (%)
Mes                                                                         
Ene          21229      569445.04  26.82    12.60  538.63     138.71  2008.3
Feb          19927      447137.35  22.44    12.75   63.06      26.16   281.0
Mar          27175      595500.76  21.91    12.60   58.53      15.19   267.1
Abr          22642      469200.36  20.72    12.60   83.25      43.39   401.8
May          28320      678594.56  23.96    14.85   79.51      47.70   331.8
Jun          27185      661213.69  24.32    12.50  247.18     144.45  1016.4
Jul          26825      600091.01  22.37    12.48   65.53      17.11   292.9
Ago          27007      645343.90  23.90    13.52   67.82      19.88   283.8
Sep          40028      952838.38  23.80    13.28   89.04      30.43   374.1
Oct          49554     1039318.79  20.97    10.50   79.56      28.66   379.4
Nov          64531     1161817.38  18.00     9.87   51.94      24.24   288.6

---
## 7. Análisis de devoluciones

Las devoluciones son una parte importante del negocio. ¿Tienen algún patrón estadístico?

In [13]:
dev = devoluciones.copy()
dev['Quantity'] = dev['Quantity'].abs()         # convertimos a positivo para analizar magnitudes
dev['Revenue']  = dev['Quantity'] * dev['UnitPrice'].abs()

print(f"Total de devoluciones: {len(dev):,} transacciones")
print(f"Revenue devuelto total: £{dev['Revenue'].sum():,.2f}")
print(f"\nPorcentaje de devoluciones sobre ventas:")
print(f"  Por transacciones: {len(dev)/len(df)*100:.1f}%")
print(f"  Por revenue:       {dev['Revenue'].sum()/df['Revenue'].sum()*100:.1f}%")

print("\n── Estadísticas del revenue devuelto ──")
print(f"  Media   : £{dev['Revenue'].mean():.2f}")
print(f"  Mediana : £{dev['Revenue'].median():.2f}")
print(f"  G1      : {dev['Revenue'].skew():.4f}")

print(f"\nTop 5 países con más devoluciones:")
print(dev.groupby('Country')['Revenue']
         .agg(n='count', total='sum')
         .sort_values('total', ascending=False)
         .head(5)
         .round(2)
         .to_string())

Total de devoluciones: 9,288 transacciones
Revenue devuelto total: £896,812.49

Porcentaje de devoluciones sobre ventas:
  Por transacciones: 2.3%
  Por revenue:       10.1%

── Estadísticas del revenue devuelto ──
  Media   : £96.56
  Mediana : £8.50
  G1      : 67.5068

Top 5 países con más devoluciones:
                   n      total
Country                        
United Kingdom  7856  815291.60
EIRE             302   20177.14
France           149   12311.21
Singapore          7   12158.90
Germany          453    7168.93


---
## 8. Correlaciones entre variables numéricas

In [14]:
print("── Matriz de correlación (Pearson) ──")
print(df[['Quantity', 'UnitPrice', 'Revenue']].corr().round(4))

print("\n── Correlación de Spearman (más robusta ante outliers) ──")
print(df[['Quantity', 'UnitPrice', 'Revenue']].corr(method='spearman').round(4))

print("\n💡 Pearson vs Spearman:")
print("   Pearson mide correlación LINEAL — sensible a outliers.")
print("   Spearman mide correlación MONÓTONA — trabaja sobre rangos, más robusta.")
print("   Con distribuciones muy asimétricas como esta, Spearman suele ser más informativo.")

── Matriz de correlación (Pearson) ──
           Quantity  UnitPrice  Revenue
Quantity     1.0000    -0.0046   0.9144
UnitPrice   -0.0046     1.0000   0.0816
Revenue      0.9144     0.0816   1.0000

── Correlación de Spearman (más robusta ante outliers) ──
           Quantity  UnitPrice  Revenue
Quantity     1.0000    -0.4077   0.6573
UnitPrice   -0.4077     1.0000   0.3487
Revenue      0.6573     0.3487   1.0000

💡 Pearson vs Spearman:
   Pearson mide correlación LINEAL — sensible a outliers.
   Spearman mide correlación MONÓTONA — trabaja sobre rangos, más robusta.
   Con distribuciones muy asimétricas como esta, Spearman suele ser más informativo.


---
## 9. Resumen

In [15]:
print("╔" + "═"*54 + "╗")
print("║   RESUMEN  — Online Retail UK              ║")
print("╠" + "═"*54 + "╣")
print(f"║  Período       : {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}  ║")
print(f"║  Transacciones : {len(df):>10,}                        ║")
print(f"║  Clientes      : {df['CustomerID'].nunique():>10,}                        ║")
print(f"║  Países        : {df['Country'].nunique():>10,}                        ║")
print(f"║  Revenue total : £{df['Revenue'].sum():>10,.2f}                  ║")
print("╠" + "═"*54 + "╣")
print("║  REVENUE POR TRANSACCIÓN                            ║")
print(f"║  Media         : £{df['Revenue'].mean():>10.2f}   (sensible a outliers) ║")
print(f"║  Mediana       : £{df['Revenue'].median():>10.2f}   (robusta)            ║")
print(f"║  Desvío        : £{df['Revenue'].std():>10.2f}                        ║")
print(f"║  CV            : {df['Revenue'].std()/df['Revenue'].mean()*100:>9.1f}%                         ║")
print(f"║  Asimetría G1  : {df['Revenue'].skew():>10.4f}   (alta asim. positiva) ║")
print("╠" + "═"*54 + "╣")
print("║  CONCLUSIONES CLAVE                                 ║")
print("║  • La media del revenue NO representa al cliente    ║")
print("║    típico → usar MEDIANA para reportes operativos   ║")
print("║  • Alto CV indica enorme variabilidad entre pedidos ║")
print("║  • Distribución de Pareto: ~20% genera ~80% revenue ║")
print("║  • UK domina el revenue; pico en nov/dic (regalos)  ║")
print("╚" + "═"*54 + "╝")

╔══════════════════════════════════════════════════════╗
║   RESUMEN  — Online Retail UK              ║
╠══════════════════════════════════════════════════════╣
║  Período       : 2010-12-01 → 2011-12-09  ║
║  Transacciones :    397,884                        ║
║  Clientes      :      4,338                        ║
║  Países        :         37                        ║
║  Revenue total : £8,911,407.90                  ║
╠══════════════════════════════════════════════════════╣
║  REVENUE POR TRANSACCIÓN                            ║
║  Media         : £     22.40   (sensible a outliers) ║
║  Mediana       : £     11.80   (robusta)            ║
║  Desvío        : £    309.07                        ║
║  CV            :    1380.0%                         ║
║  Asimetría G1  :   451.4432   (alta asim. positiva) ║
╠══════════════════════════════════════════════════════╣
║  CONCLUSIONES CLAVE                                 ║
║  • La media del revenue NO representa al cliente    ║
║    típico →

---
## ✏️ Ejercicios

**Ejercicio 1.** Calculá el revenue **promedio por cliente** (no por transacción). ¿Cómo se compara la media con la mediana? ¿Qué conclusión de negocio sacás?

In [16]:
# Solución Ejercicio 1

# 1. Calcular revenue total por cliente
revenue_por_cliente = df.groupby('CustomerID')['Revenue'].sum()

print("="*70)
print("  REVENUE PROMEDIO POR CLIENTE")
print("="*70)

# 2. Estadísticas descriptivas
print(f"\n  Total de clientes  : {len(revenue_por_cliente):,}")
print(f"  Revenue total      : £{revenue_por_cliente.sum():,.2f}")

print("\n── POSICIÓN ──")
print(f"  Media              : £{revenue_por_cliente.mean():,.2f}")
print(f"  Mediana            : £{revenue_por_cliente.median():,.2f}")
print(f"  Moda               : £{revenue_por_cliente.mode()[0] if len(revenue_por_cliente.mode()) > 0 else 'N/A'}")
print(f"  P25 / P75 / P90    : £{revenue_por_cliente.quantile(0.25):,.2f} / £{revenue_por_cliente.quantile(0.75):,.2f} / £{revenue_por_cliente.quantile(0.90):,.2f}")

print("\n── DISPERSIÓN ──")
print(f"  Mín / Máx          : £{revenue_por_cliente.min():,.2f} / £{revenue_por_cliente.max():,.2f}")
print(f"  Rango              : £{revenue_por_cliente.max() - revenue_por_cliente.min():,.2f}")
print(f"  RIQ                : £{revenue_por_cliente.quantile(0.75) - revenue_por_cliente.quantile(0.25):,.2f}")
print(f"  Desvío estándar    : £{revenue_por_cliente.std():,.2f}")
print(f"  CV                 : {revenue_por_cliente.std()/revenue_por_cliente.mean()*100:.1f}%")

print("\n── ASIMETRÍA ──")
print(f"  Fisher G1          : {revenue_por_cliente.skew():.4f}")

# 3. Comparación media vs mediana
diferencia = revenue_por_cliente.mean() - revenue_por_cliente.median()
ratio = revenue_por_cliente.mean() / revenue_por_cliente.median()

print("\n" + "="*70)
print("  COMPARACIÓN: MEDIA vs MEDIANA")
print("="*70)
print(f"  Media              : £{revenue_por_cliente.mean():,.2f}")
print(f"  Mediana            : £{revenue_por_cliente.median():,.2f}")
print(f"  Diferencia (M - Me): £{diferencia:,.2f}")
print(f"  Ratio (M / Me)     : {ratio:.2f}x")

# 4. Análisis de distribución - Regla de Pareto
clientes_ordenados = revenue_por_cliente.sort_values(ascending=False)
total_clientes = len(clientes_ordenados)
total_revenue = clientes_ordenados.sum()

print("\n" + "="*70)
print("  PRINCIPIO DE PARETO (80/20)")
print("="*70)

for pct in [0.05, 0.10, 0.20, 0.30]:
    n_top = int(total_clientes * pct)
    rev_top = clientes_ordenados.iloc[:n_top].sum()
    pct_rev = rev_top / total_revenue * 100
    print(f"  Top {pct*100:>4.0f}% de clientes ({n_top:>4,}) → {pct_rev:>5.1f}% del revenue")

# 5. Clientes de alto valor (top 10)
print("\n" + "="*70)
print("  TOP 10 CLIENTES (por revenue total)")
print("="*70)

top10_clientes = clientes_ordenados.head(10)
for idx, (cliente_id, rev) in enumerate(top10_clientes.items(), 1):
    pct_total = (rev / total_revenue) * 100
    n_transacciones = len(df[df['CustomerID'] == cliente_id])
    ticket_promedio = rev / n_transacciones
    print(f"  {idx:>2}. Cliente {int(cliente_id):>6} → £{rev:>10,.2f} ({pct_total:>4.1f}%) | {n_transacciones:>3} trans. | £{ticket_promedio:>8,.2f}/trans.")

# 6. Análisis de clientes según percentiles
print("\n" + "="*70)
print("  SEGMENTACIÓN DE CLIENTES POR REVENUE")
print("="*70)

p25 = revenue_por_cliente.quantile(0.25)
p50 = revenue_por_cliente.quantile(0.50)
p75 = revenue_por_cliente.quantile(0.75)

segmentos = [
    ("Alto valor (P75-P100)", revenue_por_cliente[revenue_por_cliente > p75]),
    ("Medio-alto (P50-P75)", revenue_por_cliente[(revenue_por_cliente > p50) & (revenue_por_cliente <= p75)]),
    ("Medio-bajo (P25-P50)", revenue_por_cliente[(revenue_por_cliente > p25) & (revenue_por_cliente <= p50)]),
    ("Bajo valor (P0-P25)", revenue_por_cliente[revenue_por_cliente <= p25])
]

for nombre, segmento in segmentos:
    n_clientes = len(segmento)
    pct_clientes = (n_clientes / total_clientes) * 100
    rev_segmento = segmento.sum()
    pct_revenue = (rev_segmento / total_revenue) * 100
    print(f"\n  {nombre}")
    print(f"    Clientes: {n_clientes:>5,} ({pct_clientes:>5.1f}%)")
    print(f"    Revenue : £{rev_segmento:>12,.2f} ({pct_revenue:>5.1f}%)")
    print(f"    Promedio: £{segmento.mean():>12,.2f}")

# 7. Conclusiones de negocio
print("\n" + "="*70)
print("  CONCLUSIONES DE NEGOCIO")
print("="*70)

print(f"""
1. ASIMETRÍA EXTREMA:
   • La media (£{revenue_por_cliente.mean():,.0f}) es {ratio:.1f}x MAYOR que la mediana (£{revenue_por_cliente.median():,.0f})
   • G1 = {revenue_por_cliente.skew():.2f} → asimetría positiva MUY fuerte
   • Esto significa que pocos clientes generan MUCHO más revenue que el típico

2. DISTRIBUCIÓN DE PARETO EXACERBADA:
   • El top 20% de clientes genera ~{(clientes_ordenados.head(int(total_clientes*0.20)).sum()/total_revenue*100):.0f}% del revenue
   • El top 10% genera ~{(clientes_ordenados.head(int(total_clientes*0.10)).sum()/total_revenue*100):.0f}% del revenue
   • Concentración MAYOR que la regla 80/20 tradicional

3. IMPLICACIONES ESTRATÉGICAS:

   a) SEGMENTACIÓN:
      • Los clientes NO son homogéneos
      • Se necesitan estrategias diferentes por segmento
      • El "cliente promedio" (media) NO EXISTE en la práctica

   b) RETENCIÓN:
      • CRÍTICO retener a los clientes de alto valor (top 20%)
      • La pérdida de un cliente top impacta significativamente
      • Programas de fidelización deben focalizarse en top clientes

   c) REPORTES Y KPIs:
      • NO usar la media como métrica de cliente típico
      • USAR la mediana (£{revenue_por_cliente.median():,.0f}) para reportar revenue típico
      • Complementar con percentiles (P25, P75, P90)

   d) ADQUISICIÓN:
      • Enfocarse en atraer clientes con potencial de alto valor
      • El costo de adquisición justificable varía MUCHO según segmento
      • Analizar características de top clientes para targeting

4. RIESGO DE CONCENTRACIÓN:
   • Alta dependencia de pocos clientes
   • El top 10 clientes representa {(top10_clientes.sum()/total_revenue*100):.1f}% del revenue
   • Diversificación de cartera de clientes como objetivo estratégico

5. MÉTRICA CORRECTA:
   • Para "cliente típico"    → MEDIANA (£{revenue_por_cliente.median():,.0f})
   • Para proyección total    → SUMA
   • Para valor de cartera    → Análisis por percentiles
   • NO usar media salvo contexto de revenue total promedio
""")

print("="*70)


  REVENUE PROMEDIO POR CLIENTE

  Total de clientes  : 4,338
  Revenue total      : £8,911,407.90

── POSICIÓN ──
  Media              : £2,054.27
  Mediana            : £674.49
  Moda               : £76.32000000000001
  P25 / P75 / P90    : £307.41 / £1,661.74 / £3,646.53

── DISPERSIÓN ──
  Mín / Máx          : £3.75 / £280,206.02
  Rango              : £280,202.27
  RIQ                : £1,354.33
  Desvío estándar    : £8,989.23
  CV                 : 437.6%

── ASIMETRÍA ──
  Fisher G1          : 19.3250

  COMPARACIÓN: MEDIA vs MEDIANA
  Media              : £2,054.27
  Mediana            : £674.49
  Diferencia (M - Me): £1,379.78
  Ratio (M / Me)     : 3.05x

  PRINCIPIO DE PARETO (80/20)
  Top    5% de clientes ( 216) →  50.3% del revenue
  Top   10% de clientes ( 433) →  61.3% del revenue
  Top   20% de clientes ( 867) →  74.6% del revenue
  Top   30% de clientes (1,301) →  82.8% del revenue

  TOP 10 CLIENTES (por revenue total)
   1. Cliente  14646 → £280,206.02 ( 3.1%) | 20

**Ejercicio 2.** Filtrá solo las transacciones del **Reino Unido** y compará la distribución de `UnitPrice` entre los meses de mayor y menor venta. ¿Hay diferencias en la asimetría?

In [17]:
# Solución Ejercicio 2

# 1. Filtrar solo transacciones del Reino Unido
df_uk = df[df['Country'] == 'United Kingdom'].copy()

print("="*70)
print("  ANÁLISIS DE PRECIO UNITARIO - REINO UNIDO")
print("="*70)
print(f"\n  Total transacciones UK: {len(df_uk):,}")
print(f"  Período: {df_uk['InvoiceDate'].min().date()} → {df_uk['InvoiceDate'].max().date()}")

# 2. Identificar meses de mayor y menor venta
meses_dict = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
              7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

revenue_por_mes = df_uk.groupby('Mes')['Revenue'].sum().sort_values(ascending=False)

mes_mayor = revenue_por_mes.idxmax()
mes_menor = revenue_por_mes.idxmin()

print("\n" + "="*70)
print("  REVENUE POR MES (UK)")
print("="*70)

for mes in revenue_por_mes.index:
    nombre_mes = meses_dict[mes]
    rev = revenue_por_mes[mes]
    marcador = ""
    if mes == mes_mayor:
        marcador = " ← MAYOR"
    elif mes == mes_menor:
        marcador = " ← MENOR"
    print(f"  {nombre_mes:>3} : £{rev:>12,.2f}{marcador}")

# 3. Filtrar datos de ambos meses
precios_mes_mayor = df_uk[df_uk['Mes'] == mes_mayor]['UnitPrice']
precios_mes_menor = df_uk[df_uk['Mes'] == mes_menor]['UnitPrice']

# 4. Análisis estadístico comparativo
print("\n" + "="*70)
print(f"  COMPARACIÓN: {meses_dict[mes_mayor]} (mayor venta) vs {meses_dict[mes_menor]} (menor venta)")
print("="*70)

comparacion_data = {
    meses_dict[mes_mayor]: precios_mes_mayor,
    meses_dict[mes_menor]: precios_mes_menor
}

print(f"\n{'Estadística':<22} {meses_dict[mes_mayor]:>15} {meses_dict[mes_menor]:>15} {'Diferencia':>15}")
print("─" * 70)

# Métricas de posición
print(f"{'n transacciones':<22} {len(precios_mes_mayor):>15,} {len(precios_mes_menor):>15,} {len(precios_mes_mayor) - len(precios_mes_menor):>15,}")
print(f"{'Media (£)':<22} {precios_mes_mayor.mean():>15.2f} {precios_mes_menor.mean():>15.2f} {precios_mes_mayor.mean() - precios_mes_menor.mean():>+15.2f}")
print(f"{'Mediana (£)':<22} {precios_mes_mayor.median():>15.2f} {precios_mes_menor.median():>15.2f} {precios_mes_mayor.median() - precios_mes_menor.median():>+15.2f}")
print(f"{'Moda (£)':<22} {precios_mes_mayor.mode()[0]:>15.2f} {precios_mes_menor.mode()[0]:>15.2f} {precios_mes_mayor.mode()[0] - precios_mes_menor.mode()[0]:>+15.2f}")

# Percentiles
print(f"{'Q1 (£)':<22} {precios_mes_mayor.quantile(0.25):>15.2f} {precios_mes_menor.quantile(0.25):>15.2f} {precios_mes_mayor.quantile(0.25) - precios_mes_menor.quantile(0.25):>+15.2f}")
print(f"{'Q3 (£)':<22} {precios_mes_mayor.quantile(0.75):>15.2f} {precios_mes_menor.quantile(0.75):>15.2f} {precios_mes_mayor.quantile(0.75) - precios_mes_menor.quantile(0.75):>+15.2f}")
print(f"{'P90 (£)':<22} {precios_mes_mayor.quantile(0.90):>15.2f} {precios_mes_menor.quantile(0.90):>15.2f} {precios_mes_mayor.quantile(0.90) - precios_mes_menor.quantile(0.90):>+15.2f}")

# Métricas de dispersión
print(f"{'Desvío std (£)':<22} {precios_mes_mayor.std():>15.2f} {precios_mes_menor.std():>15.2f} {precios_mes_mayor.std() - precios_mes_menor.std():>+15.2f}")
print(f"{'CV (%)':<22} {(precios_mes_mayor.std()/precios_mes_mayor.mean()*100):>15.1f} {(precios_mes_menor.std()/precios_mes_menor.mean()*100):>15.1f} {(precios_mes_mayor.std()/precios_mes_mayor.mean()*100) - (precios_mes_menor.std()/precios_mes_menor.mean()*100):>+15.1f}")
print(f"{'RIQ (£)':<22} {(precios_mes_mayor.quantile(0.75)-precios_mes_mayor.quantile(0.25)):>15.2f} {(precios_mes_menor.quantile(0.75)-precios_mes_menor.quantile(0.25)):>15.2f} {(precios_mes_mayor.quantile(0.75)-precios_mes_mayor.quantile(0.25)) - (precios_mes_menor.quantile(0.75)-precios_mes_menor.quantile(0.25)):>+15.2f}")

# Asimetría - EL FOCO DEL EJERCICIO
print("\n" + "─" * 70)
print(f"{'ASIMETRÍA':<22}")
print("─" * 70)
print(f"{'Fisher G1':<22} {precios_mes_mayor.skew():>15.4f} {precios_mes_menor.skew():>15.4f} {precios_mes_mayor.skew() - precios_mes_menor.skew():>+15.4f}")
print(f"{'Pearson (Me)':<22} {3*(precios_mes_mayor.mean()-precios_mes_mayor.median())/precios_mes_mayor.std():>15.4f} {3*(precios_mes_menor.mean()-precios_mes_menor.median())/precios_mes_menor.std():>15.4f} {(3*(precios_mes_mayor.mean()-precios_mes_mayor.median())/precios_mes_mayor.std()) - (3*(precios_mes_menor.mean()-precios_mes_menor.median())/precios_mes_menor.std()):>+15.4f}")

# 5. Interpretación detallada de asimetría
print("\n" + "="*70)
print("  ANÁLISIS DETALLADO DE ASIMETRÍA")
print("="*70)

g1_mayor = precios_mes_mayor.skew()
g1_menor = precios_mes_menor.skew()
diff_g1 = abs(g1_mayor - g1_menor)

print(f"\n{meses_dict[mes_mayor].upper()} (mes de MAYOR venta):")
print(f"  G1 = {g1_mayor:.4f}")
if g1_mayor > 0.5:
    tipo_mayor = "Asimetría positiva FUERTE (cola derecha)"
elif g1_mayor > 0:
    tipo_mayor = "Asimetría positiva MODERADA (cola derecha)"
elif g1_mayor > -0.5:
    tipo_mayor = "Aproximadamente simétrica"
else:
    tipo_mayor = "Asimetría negativa (cola izquierda)"
print(f"  Tipo: {tipo_mayor}")
print(f"  Media = £{precios_mes_mayor.mean():.2f}  |  Mediana = £{precios_mes_mayor.median():.2f}")
print(f"  Diferencia Media-Mediana: £{precios_mes_mayor.mean() - precios_mes_mayor.median():.2f}")

print(f"\n{meses_dict[mes_menor].upper()} (mes de MENOR venta):")
print(f"  G1 = {g1_menor:.4f}")
if g1_menor > 0.5:
    tipo_menor = "Asimetría positiva FUERTE (cola derecha)"
elif g1_menor > 0:
    tipo_menor = "Asimetría positiva MODERADA (cola derecha)"
elif g1_menor > -0.5:
    tipo_menor = "Aproximadamente simétrica"
else:
    tipo_menor = "Asimetría negativa (cola izquierda)"
print(f"  Tipo: {tipo_menor}")
print(f"  Media = £{precios_mes_menor.mean():.2f}  |  Mediana = £{precios_mes_menor.median():.2f}")
print(f"  Diferencia Media-Mediana: £{precios_mes_menor.mean() - precios_mes_menor.median():.2f}")

# 6. Conclusiones
print("\n" + "="*70)
print("  CONCLUSIONES")
print("="*70)

print(f"""
1. DIFERENCIA EN ASIMETRÍA:
   • {meses_dict[mes_mayor]}: G1 = {g1_mayor:.4f}
   • {meses_dict[mes_menor]}: G1 = {g1_menor:.4f}
   • Diferencia absoluta: {diff_g1:.4f}
   {'   → DIFERENCIA SIGNIFICATIVA' if diff_g1 > 0.1 else '   → Diferencia moderada' if diff_g1 > 0.05 else '   → Diferencia pequeña'}

2. INTERPRETACIÓN:
   • Ambos meses muestran {'asimetría positiva' if g1_mayor > 0 and g1_menor > 0 else 'patrones diferentes'}
   • {meses_dict[mes_mayor]} tiene {'MAYOR' if abs(g1_mayor) > abs(g1_menor) else 'MENOR'} asimetría que {meses_dict[mes_menor]}
   • {'Más productos de alto precio' if g1_mayor > g1_menor else 'Distribución más equilibrada'} en {meses_dict[mes_mayor if g1_mayor > g1_menor else mes_menor]}

3. CONTEXTO DE NEGOCIO:
   • {meses_dict[mes_mayor]} es {'típicamente temporada alta (fin de año/regalos)' if mes_mayor in [11, 12] else 'mes de alta demanda'}
   • {meses_dict[mes_menor]} es {'mes de baja temporada' if mes_menor in [1, 2] else 'período de menor actividad'}
   • {'Las diferencias en asimetría sugieren mix de productos diferente' if diff_g1 > 0.1 else 'El mix de productos es similar entre meses'}

4. HALLAZGOS CLAVE:
   • {'El mes de mayor venta NO significa precios más altos en promedio' if precios_mes_mayor.mean() < precios_mes_menor.mean() else 'Mayor volumen de ventas con precios elevados'}
   • Coeficiente de variación: {meses_dict[mes_mayor]} = {(precios_mes_mayor.std()/precios_mes_mayor.mean()*100):.1f}%, {meses_dict[mes_menor]} = {(precios_mes_menor.std()/precios_mes_menor.mean()*100):.1f}%
   • {'Alta dispersión de precios en ambos meses' if (precios_mes_mayor.std()/precios_mes_mayor.mean()*100) > 100 else 'Dispersión moderada de precios'}
   • La mediana es más estable que la media entre meses (robusta ante outliers)

5. RECOMENDACIÓN:
   • Usar mediana para comparar precios "típicos" entre meses
   • Analizar percentiles (P25, P50, P75) para entender rango de productos
   • {'Investigar productos de alto precio en ' + meses_dict[mes_mayor if g1_mayor > g1_menor else mes_menor] + ' (generan la asimetría)' if diff_g1 > 0.1 else 'Distribución de precios es consistente'}
""")

print("="*70)

# 7. Análisis adicional: distribución de productos por rango de precio
print("\n" + "="*70)
print("  DISTRIBUCIÓN POR RANGOS DE PRECIO")
print("="*70)

rangos = [(0, 1), (1, 2), (2, 5), (5, 10), (10, 20), (20, float('inf'))]
print(f"\n{'Rango (£)':<20} {meses_dict[mes_mayor]+' (%)':>15} {meses_dict[mes_menor]+' (%)':>15}")
print("─" * 52)

for min_p, max_p in rangos:
    if max_p == float('inf'):
        rango_str = f"£{min_p}+"
        count_mayor = len(precios_mes_mayor[precios_mes_mayor >= min_p])
        count_menor = len(precios_mes_menor[precios_mes_menor >= min_p])
    else:
        rango_str = f"£{min_p} - £{max_p}"
        count_mayor = len(precios_mes_mayor[(precios_mes_mayor >= min_p) & (precios_mes_mayor < max_p)])
        count_menor = len(precios_mes_menor[(precios_mes_menor >= min_p) & (precios_mes_menor < max_p)])
    
    pct_mayor = (count_mayor / len(precios_mes_mayor)) * 100
    pct_menor = (count_menor / len(precios_mes_menor)) * 100
    
    print(f"{rango_str:<20} {pct_mayor:>14.1f}% {pct_menor:>14.1f}%")

print("\n💡 Esta tabla muestra cómo se distribuyen los precios en diferentes rangos,")
print("   ayudando a entender qué causa las diferencias en asimetría.")


  ANÁLISIS DE PRECIO UNITARIO - REINO UNIDO

  Total transacciones UK: 354,321
  Período: 2010-12-01 → 2011-12-09

  REVENUE POR MES (UK)
  Nov : £  980,645.75 ← MAYOR
  Dic : £  971,046.02
  Oct : £  824,766.22
  Sep : £  796,780.27
  May : £  551,568.82
  Jun : £  524,915.48
  Ago : £  498,453.32
  Jul : £  485,612.25
  Mar : £  467,198.59
  Ene : £  442,190.06
  Abr : £  409,559.14
  Feb : £  355,655.63 ← MENOR

  COMPARACIÓN: Nov (mayor venta) vs Feb (menor venta)

Estadística                        Nov             Feb      Diferencia
──────────────────────────────────────────────────────────────────────
n transacciones                 58,800          17,758          41,042
Media (£)                         2.78            3.09           -0.31
Mediana (£)                       1.69            1.95           -0.26
Moda (£)                          1.25            1.25           +0.00
Q1 (£)                            1.25            1.25           +0.00
Q3 (£)                       

**Ejercicio 3.** Encontrá los **10 productos más vendidos** (por quantity total) y calculá su precio promedio, mediana y CV. ¿Los productos más vendidos son los más baratos?

In [18]:
# Solución Ejercicio 3

# 1. Calcular cantidad total vendida por producto
ventas_por_producto = df.groupby('StockCode').agg({
    'Quantity': 'sum',
    'UnitPrice': ['mean', 'median', 'std', 'count'],
    'Description': 'first',  # Tomamos la primera descripción (pueden variar ligeramente)
    'Revenue': 'sum'
}).round(2)

# Aplanar columnas multi-nivel
ventas_por_producto.columns = ['Quantity_Total', 'Precio_Promedio', 'Precio_Mediana', 
                                'Precio_Std', 'N_Transacciones', 'Descripcion', 'Revenue_Total']

# Calcular CV (%)
ventas_por_producto['CV_Precio'] = (ventas_por_producto['Precio_Std'] / 
                                     ventas_por_producto['Precio_Promedio'] * 100).round(1)

# 2. Top 10 productos más vendidos
top10_productos = ventas_por_producto.nlargest(10, 'Quantity_Total')

print("="*90)
print("  TOP 10 PRODUCTOS MÁS VENDIDOS (por cantidad total)")
print("="*90)
print(f"\n{'Rank':<5} {'StockCode':<12} {'Descripción':<35} {'Qty Total':>10}")
print("─" * 90)

for idx, (stock_code, row) in enumerate(top10_productos.iterrows(), 1):
    desc = str(row['Descripcion'])[:34]  # Truncar descripción larga
    print(f"{idx:<5} {stock_code:<12} {desc:<35} {row['Quantity_Total']:>10,.0f}")

# 3. Análisis estadístico detallado de precios
print("\n" + "="*90)
print("  ANÁLISIS DE PRECIOS - TOP 10 PRODUCTOS")
print("="*90)
print(f"\n{'Rank':<5} {'StockCode':<12} {'Promedio':>10} {'Mediana':>10} {'Desvío':>10} {'CV (%)':>8} {'N Trans':>8}")
print("─" * 90)

for idx, (stock_code, row) in enumerate(top10_productos.iterrows(), 1):
    print(f"{idx:<5} {stock_code:<12} £{row['Precio_Promedio']:>9.2f} £{row['Precio_Mediana']:>9.2f} "
          f"£{row['Precio_Std']:>9.2f} {row['CV_Precio']:>7.1f}% {row['N_Transacciones']:>8,.0f}")

# 4. Estadísticas agregadas del top 10
print("\n" + "="*90)
print("  RESUMEN ESTADÍSTICO - TOP 10")
print("="*90)

precios_top10 = top10_productos['Precio_Promedio']
cantidades_top10 = top10_productos['Quantity_Total']

print(f"\nPRECIO PROMEDIO (de los promedios):")
print(f"  Media           : £{precios_top10.mean():.2f}")
print(f"  Mediana         : £{precios_top10.median():.2f}")
print(f"  Mín / Máx       : £{precios_top10.min():.2f} / £{precios_top10.max():.2f}")
print(f"  Desvío std      : £{precios_top10.std():.2f}")
print(f"  CV              : {(precios_top10.std()/precios_top10.mean()*100):.1f}%")

print(f"\nCANTIDAD VENDIDA:")
print(f"  Media           : {cantidades_top10.mean():,.0f} unidades")
print(f"  Mediana         : {cantidades_top10.median():,.0f} unidades")
print(f"  Total Top 10    : {cantidades_top10.sum():,.0f} unidades")
print(f"  % del total     : {(cantidades_top10.sum()/df['Quantity'].sum()*100):.1f}%")

# 5. Análisis de correlación: Precio vs Cantidad
from scipy.stats import pearsonr, spearmanr

corr_pearson, p_pearson = pearsonr(top10_productos['Precio_Promedio'], 
                                     top10_productos['Quantity_Total'])
corr_spearman, p_spearman = spearmanr(top10_productos['Precio_Promedio'], 
                                       top10_productos['Quantity_Total'])

print("\n" + "="*90)
print("  CORRELACIÓN: PRECIO vs CANTIDAD VENDIDA")
print("="*90)
print(f"\n  Correlación de Pearson  : {corr_pearson:>6.4f}  (p-value: {p_pearson:.4f})")
print(f"  Correlación de Spearman : {corr_spearman:>6.4f}  (p-value: {p_spearman:.4f})")

# Interpretación de correlación
if abs(corr_pearson) < 0.3:
    interpretacion_corr = "Correlación DÉBIL o inexistente"
elif abs(corr_pearson) < 0.7:
    interpretacion_corr = "Correlación MODERADA"
else:
    interpretacion_corr = "Correlación FUERTE"

direccion = "NEGATIVA (a menor precio, mayor cantidad)" if corr_pearson < 0 else "POSITIVA (a mayor precio, mayor cantidad)"

print(f"\n  Interpretación: {interpretacion_corr}")
print(f"  Dirección     : {direccion}")

# 6. Comparación con todos los productos
print("\n" + "="*90)
print("  COMPARACIÓN: TOP 10 vs TODOS LOS PRODUCTOS")
print("="*90)

# Precio promedio de TODOS los productos (agrupados)
precios_todos = df.groupby('StockCode')['UnitPrice'].mean()

print(f"\n{'Métrica':<30} {'Top 10':>15} {'Todos':>15} {'Diferencia':>15}")
print("─" * 77)
print(f"{'N productos':<30} {len(top10_productos):>15,} {len(precios_todos):>15,} {len(top10_productos) - len(precios_todos):>15,}")
print(f"{'Precio promedio (£)':<30} {precios_top10.mean():>15.2f} {precios_todos.mean():>15.2f} {precios_top10.mean() - precios_todos.mean():>+15.2f}")
print(f"{'Precio mediano (£)':<30} {precios_top10.median():>15.2f} {precios_todos.median():>15.2f} {precios_top10.median() - precios_todos.median():>+15.2f}")

diferencia_pct = ((precios_top10.mean() - precios_todos.mean()) / precios_todos.mean() * 100)
print(f"\n💡 Los productos más vendidos son {abs(diferencia_pct):.1f}% {'MÁS BARATOS' if diferencia_pct < 0 else 'MÁS CAROS'} que el promedio general")

# 7. Clasificación de productos por precio
print("\n" + "="*90)
print("  CLASIFICACIÓN DE TOP 10 POR RANGO DE PRECIO")
print("="*90)

precio_mediano_general = precios_todos.median()
precio_promedio_general = precios_todos.mean()

print(f"\nReferencia - Precio mediano general: £{precio_mediano_general:.2f}")
print(f"Referencia - Precio promedio general: £{precio_promedio_general:.2f}\n")

baratos = top10_productos[top10_productos['Precio_Promedio'] < precio_mediano_general]
medios = top10_productos[(top10_productos['Precio_Promedio'] >= precio_mediano_general) & 
                         (top10_productos['Precio_Promedio'] < precio_promedio_general * 2)]
caros = top10_productos[top10_productos['Precio_Promedio'] >= precio_promedio_general * 2]

print(f"{'Categoría':<20} {'N productos':>12} {'% del Top 10':>14} {'Qty promedio':>14}")
print("─" * 62)
print(f"{'Baratos (< mediana)':<20} {len(baratos):>12} {len(baratos)/10*100:>13.0f}% {baratos['Quantity_Total'].mean() if len(baratos) > 0 else 0:>13,.0f}")
print(f"{'Medios':<20} {len(medios):>12} {len(medios)/10*100:>13.0f}% {medios['Quantity_Total'].mean() if len(medios) > 0 else 0:>13,.0f}")
print(f"{'Caros (> 2×promedio)':<20} {len(caros):>12} {len(caros)/10*100:>13.0f}% {caros['Quantity_Total'].mean() if len(caros) > 0 else 0:>13,.0f}")

# 8. Análisis por descripción de producto
print("\n" + "="*90)
print("  DETALLE COMPLETO - TOP 10 PRODUCTOS")
print("="*90)

for idx, (stock_code, row) in enumerate(top10_productos.iterrows(), 1):
    print(f"\n{idx}. {stock_code} — {row['Descripcion']}")
    print(f"   Cantidad vendida  : {row['Quantity_Total']:>10,.0f} unidades")
    print(f"   Revenue generado  : £{row['Revenue_Total']:>10,.2f}")
    print(f"   Precio promedio   : £{row['Precio_Promedio']:>10.2f}")
    print(f"   Precio mediana    : £{row['Precio_Mediana']:>10.2f}")
    print(f"   Desvío std precio : £{row['Precio_Std']:>10.2f}")
    print(f"   CV                : {row['CV_Precio']:>9.1f}%")
    print(f"   Transacciones     : {row['N_Transacciones']:>10,.0f}")
    
    # Clasificación
    if row['Precio_Promedio'] < precio_mediano_general:
        categoria = "BARATO"
    elif row['Precio_Promedio'] < precio_promedio_general * 2:
        categoria = "MEDIO"
    else:
        categoria = "CARO"
    print(f"   Categoría         : {categoria}")

# 9. Conclusiones de negocio
print("\n" + "="*90)
print("  CONCLUSIONES")
print("="*90)

print(f"""
1. ¿LOS MÁS VENDIDOS SON LOS MÁS BARATOS?
   {'→ SÍ' if diferencia_pct < -10 else '→ NO necesariamente' if abs(diferencia_pct) < 10 else '→ NO, son más caros'}
   
   • Precio promedio Top 10: £{precios_top10.mean():.2f}
   • Precio promedio general: £{precios_todos.mean():.2f}
   • Diferencia: {diferencia_pct:+.1f}%
   
   {'   Los productos más vendidos SÍ tienden a ser más baratos que el promedio.' if diferencia_pct < -10 else 
    '   No hay una relación clara: hay productos baratos y caros entre los más vendidos.' if abs(diferencia_pct) < 10 else
    '   Los productos más vendidos son MÁS CAROS que el promedio (productos premium de alto volumen).'}

2. CORRELACIÓN PRECIO-VOLUMEN:
   • Pearson: {corr_pearson:.4f} → {interpretacion_corr}
   • {'La cantidad vendida NO está determinada principalmente por el precio' if abs(corr_pearson) < 0.3 else
      'Hay una relación moderada entre precio y volumen' if abs(corr_pearson) < 0.7 else
      'Hay una relación fuerte entre precio y volumen'}
   • Otros factores (utilidad, estacionalidad, marketing) son igual o más importantes

3. CONCENTRACIÓN DE VENTAS:
   • Los 10 productos más vendidos representan {(cantidades_top10.sum()/df['Quantity'].sum()*100):.1f}% del volumen total
   • {'Alta concentración' if (cantidades_top10.sum()/df['Quantity'].sum()*100) > 20 else 'Concentración moderada'}
   • Gestión de stock crítica para estos productos

4. VARIABILIDAD DE PRECIOS (CV):
   • CV promedio: {top10_productos['CV_Precio'].mean():.1f}%
   • {'Precios muy consistentes' if top10_productos['CV_Precio'].mean() < 10 else
      'Precios relativamente estables' if top10_productos['CV_Precio'].mean() < 30 else
      'Alta variabilidad de precios (descuentos, promociones, o errores de carga)'}
   • Los productos con CV > 30% necesitan revisión de política de precios

5. DISTRIBUCIÓN DE CATEGORÍAS:
   • Productos baratos    : {len(baratos)}/10
   • Productos precio medio: {len(medios)}/10
   • Productos caros      : {len(caros)}/10
   
   {'   → Mix diverso: el volumen no depende solo del precio bajo' if len(baratos) < 7 else
    '   → Dominan productos baratos: estrategia de volumen con bajo margen'}

6. IMPLICACIONES ESTRATÉGICAS:

   a) GESTIÓN DE INVENTARIO:
      • Priorizar stock de estos 10 productos (evitar quiebres)
      • Son productos de alta rotación → cash flow positivo
      
   b) PRICING:
      • {'Hay espacio para aumentos de precio controlados en productos baratos de alto volumen' if diferencia_pct < -20 else
         'Mantener precios competitivos en productos clave' if abs(diferencia_pct) < 10 else
         'Cuidado con subir precios en productos ya premium'}
      • Analizar elasticidad precio-demanda para cada SKU
      
   c) MARKETING:
      • Estos productos son "bestsellers" → usar en campañas
      • Bundling: combinar con productos de menor rotación
      • Cross-selling: productos complementarios
      
   d) RENTABILIDAD:
      • No confundir volumen con rentabilidad
      • Calcular margen por producto (requiere costo)
      • {'Productos baratos de alto volumen pueden tener margen bajo' if diferencia_pct < -10 else
         'Verificar si alto precio compensa menor volumen relativo'}

7. PRÓXIMOS PASOS:
   • Análisis ABC: clasificar TODOS los productos por revenue
   • Análisis de margen: volumen × (precio - costo)
   • Análisis de elasticidad: ¿qué pasa si cambiamos precios?
   • Segmentación por cliente: ¿quiénes compran estos productos?
""")

print("="*90)

  TOP 10 PRODUCTOS MÁS VENDIDOS (por cantidad total)

Rank  StockCode    Descripción                          Qty Total
──────────────────────────────────────────────────────────────────────────────────────────
1     23843        PAPER CRAFT , LITTLE BIRDIE             80,995
2     23166        MEDIUM CERAMIC TOP STORAGE JAR          77,916
3     84077        WORLD WAR 2 GLIDERS ASSTD DESIGNS       54,415
4     22197        SMALL POPCORN HOLDER                    49,183
5     85099B       JUMBO BAG RED RETROSPOT                 46,181
6     85123A       WHITE HANGING HEART T-LIGHT HOLDER      36,782
7     84879        ASSORTED COLOUR BIRD ORNAMENT           35,362
8     21212        PACK OF 72 RETROSPOT CAKE CASES         33,693
9     23084        RABBIT NIGHT LIGHT                      27,202
10    22492        MINI PAINT SET VINTAGE                  26,076

  ANÁLISIS DE PRECIOS - TOP 10 PRODUCTOS

Rank  StockCode      Promedio    Mediana     Desvío   CV (%)  N Trans
────────────────

**Ejercicio 4.** Calculá el **ticket promedio por día de semana** (revenue total / cantidad de transacciones). ¿Qué día tiene mayor ticket promedio? ¿Coincide con la mediana?

In [19]:
# Solución Ejercicio 4

# 1. Análisis del ticket promedio por día de semana
print("="*80)
print("  TICKET PROMEDIO POR DÍA DE SEMANA")
print("="*80)

# Agrupar por día de semana
resumen_dias = df.groupby('DiaSemana')['Revenue'].agg(
    transacciones='count',
    revenue_total='sum',
    ticket_promedio='mean',   # media del revenue por transacción
    ticket_mediano='median',  # mediana del revenue por transacción
    desvio='std',
    asimetria='skew'
).round(2)

# Calcular CV
resumen_dias['CV (%)'] = (resumen_dias['desvio'] / resumen_dias['ticket_promedio'] * 100).round(1)

# Ordenar por días de la semana (lunes a domingo)
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
# Filtrar solo los días que existen en el dataset
orden_dias_existentes = [dia for dia in orden_dias if dia in resumen_dias.index]
resumen_dias = resumen_dias.reindex(orden_dias_existentes)

# Traducción al español
traduccion_dias = {
    'Monday': 'Lunes',
    'Tuesday': 'Martes',
    'Wednesday': 'Miércoles',
    'Thursday': 'Jueves',
    'Friday': 'Viernes',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

# Crear versión con nombres en español para mostrar
resumen_dias_esp = resumen_dias.copy()
resumen_dias_esp.index = [traduccion_dias.get(dia, dia) for dia in resumen_dias_esp.index]

print(f"\n{'Día':<12} {'Trans.':>10} {'Revenue Total':>15} {'Ticket Prom.':>13} {'Ticket Med.':>13}")
print("─" * 80)

for dia_esp, dia_eng in zip(resumen_dias_esp.index, resumen_dias.index):
    row = resumen_dias.loc[dia_eng]
    print(f"{dia_esp:<12} {row['transacciones']:>10,.0f} £{row['revenue_total']:>14,.2f} "
          f"£{row['ticket_promedio']:>12.2f} £{row['ticket_mediano']:>12.2f}")

# 2. Identificar día con mayor ticket promedio y mediano
dia_max_promedio = resumen_dias['ticket_promedio'].idxmax()
dia_max_mediano = resumen_dias['ticket_mediano'].idxmax()

dia_min_promedio = resumen_dias['ticket_promedio'].idxmin()
dia_min_mediano = resumen_dias['ticket_mediano'].idxmin()

print("\n" + "="*80)
print("  ANÁLISIS COMPARATIVO")
print("="*80)

print(f"\nDía con MAYOR ticket promedio: {traduccion_dias[dia_max_promedio]} (£{resumen_dias.loc[dia_max_promedio, 'ticket_promedio']:.2f})")
print(f"Día con MAYOR ticket mediano : {traduccion_dias[dia_max_mediano]} (£{resumen_dias.loc[dia_max_mediano, 'ticket_mediano']:.2f})")

if dia_max_promedio == dia_max_mediano:
    print(f"\n✓ SÍ COINCIDEN: {traduccion_dias[dia_max_promedio]} tiene el mayor ticket tanto en promedio como en mediana")
else:
    print(f"\n✗ NO COINCIDEN:")
    print(f"  • Mayor promedio: {traduccion_dias[dia_max_promedio]}")
    print(f"  • Mayor mediana : {traduccion_dias[dia_max_mediano]}")
    print(f"  → Esto sugiere que {traduccion_dias[dia_max_promedio]} tiene transacciones de alto valor que elevan la media")

print(f"\nDía con MENOR ticket promedio: {traduccion_dias[dia_min_promedio]} (£{resumen_dias.loc[dia_min_promedio, 'ticket_promedio']:.2f})")
print(f"Día con MENOR ticket mediano : {traduccion_dias[dia_min_mediano]} (£{resumen_dias.loc[dia_min_mediano, 'ticket_mediano']:.2f})")

# 3. Análisis estadístico detallado
print("\n" + "="*80)
print("  ESTADÍSTICAS COMPLETAS POR DÍA")
print("="*80)

print(f"\n{'Día':<12} {'Desvío (£)':>12} {'CV (%)':>8} {'G1':>8} {'Ratio M/Me':>12}")
print("─" * 80)

for dia_esp, dia_eng in zip(resumen_dias_esp.index, resumen_dias.index):
    row = resumen_dias.loc[dia_eng]
    ratio = row['ticket_promedio'] / row['ticket_mediano']
    print(f"{dia_esp:<12} £{row['desvio']:>11.2f} {row['CV (%)']:>7.1f}% {row['asimetria']:>7.4f} {ratio:>11.2f}x")

# 4. Comparación detallada entre día mayor y menor
print("\n" + "="*80)
print(f"  COMPARACIÓN: {traduccion_dias[dia_max_promedio].upper()} vs {traduccion_dias[dia_min_promedio].upper()}")
print("="*80)

mayor = resumen_dias.loc[dia_max_promedio]
menor = resumen_dias.loc[dia_min_promedio]

print(f"\n{'Métrica':<25} {traduccion_dias[dia_max_promedio]:>15} {traduccion_dias[dia_min_promedio]:>15} {'Diferencia':>15}")
print("─" * 78)
print(f"{'Transacciones':<25} {mayor['transacciones']:>15,.0f} {menor['transacciones']:>15,.0f} {mayor['transacciones']-menor['transacciones']:>+15,.0f}")
print(f"{'Revenue total (£)':<25} {mayor['revenue_total']:>15,.2f} {menor['revenue_total']:>15,.2f} {mayor['revenue_total']-menor['revenue_total']:>+15,.2f}")
print(f"{'Ticket promedio (£)':<25} {mayor['ticket_promedio']:>15.2f} {menor['ticket_promedio']:>15.2f} {mayor['ticket_promedio']-menor['ticket_promedio']:>+15.2f}")
print(f"{'Ticket mediano (£)':<25} {mayor['ticket_mediano']:>15.2f} {menor['ticket_mediano']:>15.2f} {mayor['ticket_mediano']-menor['ticket_mediano']:>+15.2f}")
print(f"{'Desvío (£)':<25} {mayor['desvio']:>15.2f} {menor['desvio']:>15.2f} {mayor['desvio']-menor['desvio']:>+15.2f}")
print(f"{'CV (%)':<25} {mayor['CV (%)']:>15.1f} {menor['CV (%)']:>15.1f} {mayor['CV (%)']-menor['CV (%)']:>+15.1f}")
print(f"{'Asimetría G1':<25} {mayor['asimetria']:>15.4f} {menor['asimetria']:>15.4f} {mayor['asimetria']-menor['asimetria']:>+15.4f}")

# Diferencia porcentual
dif_pct_promedio = ((mayor['ticket_promedio'] - menor['ticket_promedio']) / menor['ticket_promedio'] * 100)
dif_pct_mediano = ((mayor['ticket_mediano'] - menor['ticket_mediano']) / menor['ticket_mediano'] * 100)

print(f"\n💡 El ticket promedio en {traduccion_dias[dia_max_promedio]} es {dif_pct_promedio:+.1f}% {'mayor' if dif_pct_promedio > 0 else 'menor'} que en {traduccion_dias[dia_min_promedio]}")
print(f"   El ticket mediano en {traduccion_dias[dia_max_promedio]} es {dif_pct_mediano:+.1f}% {'mayor' if dif_pct_mediano > 0 else 'menor'} que en {traduccion_dias[dia_min_promedio]}")

# 5. Variabilidad entre días
print("\n" + "="*80)
print("  VARIABILIDAD ENTRE DÍAS DE SEMANA")
print("="*80)

tickets_promedio = resumen_dias['ticket_promedio']
tickets_mediano = resumen_dias['ticket_mediano']

print(f"\nTicket promedio entre días:")
print(f"  Media           : £{tickets_promedio.mean():.2f}")
print(f"  Mediana         : £{tickets_promedio.median():.2f}")
print(f"  Mín / Máx       : £{tickets_promedio.min():.2f} / £{tickets_promedio.max():.2f}")
print(f"  Rango           : £{tickets_promedio.max() - tickets_promedio.min():.2f}")
print(f"  Desvío std      : £{tickets_promedio.std():.2f}")
print(f"  CV              : {(tickets_promedio.std()/tickets_promedio.mean()*100):.1f}%")

variabilidad_cv = (tickets_promedio.std()/tickets_promedio.mean()*100)
if variabilidad_cv < 5:
    interpretacion = "MUY BAJA → tickets consistentes entre días"
elif variabilidad_cv < 10:
    interpretacion = "BAJA → ligeras diferencias entre días"
elif variabilidad_cv < 20:
    interpretacion = "MODERADA → diferencias notables entre días"
else:
    interpretacion = "ALTA → diferencias significativas entre días"

print(f"\n  Interpretación: Variabilidad {interpretacion}")

# 6. Distribución de transacciones por día
print("\n" + "="*80)
print("  DISTRIBUCIÓN DE TRANSACCIONES Y REVENUE")
print("="*80)

total_trans = resumen_dias['transacciones'].sum()
total_rev = resumen_dias['revenue_total'].sum()

print(f"\n{'Día':<12} {'Trans.':>10} {'% Trans':>8} {'Revenue':>15} {'% Revenue':>10}")
print("─" * 80)

for dia_esp, dia_eng in zip(resumen_dias_esp.index, resumen_dias.index):
    row = resumen_dias.loc[dia_eng]
    pct_trans = (row['transacciones'] / total_trans * 100)
    pct_rev = (row['revenue_total'] / total_rev * 100)
    print(f"{dia_esp:<12} {row['transacciones']:>10,.0f} {pct_trans:>7.1f}% £{row['revenue_total']:>14,.2f} {pct_rev:>9.1f}%")

# 7. Análisis de asimetría por día
print("\n" + "="*80)
print("  ANÁLISIS DE ASIMETRÍA POR DÍA")
print("="*80)

print(f"\n{'Día':<12} {'G1':>8} {'Tipo de asimetría':<30} {'Ratio M/Me':>12}")
print("─" * 80)

for dia_esp, dia_eng in zip(resumen_dias_esp.index, resumen_dias.index):
    row = resumen_dias.loc[dia_eng]
    g1 = row['asimetria']
    ratio = row['ticket_promedio'] / row['ticket_mediano']
    
    if g1 > 0.5:
        tipo = "Positiva FUERTE (cola derecha)"
    elif g1 > 0:
        tipo = "Positiva moderada"
    elif g1 > -0.5:
        tipo = "Aproximadamente simétrica"
    else:
        tipo = "Negativa (cola izquierda)"
    
    print(f"{dia_esp:<12} {g1:>7.4f} {tipo:<30} {ratio:>11.2f}x")

print("\n💡 Todos los días muestran asimetría positiva → presencia de transacciones de alto valor")
print("   El ratio Media/Mediana > 1 confirma la cola derecha en la distribución")

# 8. Ranking de días
print("\n" + "="*80)
print("  RANKING DE DÍAS")
print("="*80)

print("\nPor TICKET PROMEDIO (mayor a menor):")
ranking_promedio = resumen_dias.sort_values('ticket_promedio', ascending=False)
for idx, (dia_eng, row) in enumerate(ranking_promedio.iterrows(), 1):
    print(f"  {idx}. {traduccion_dias[dia_eng]:<12} £{row['ticket_promedio']:.2f}")

print("\nPor TICKET MEDIANO (mayor a menor):")
ranking_mediano = resumen_dias.sort_values('ticket_mediano', ascending=False)
for idx, (dia_eng, row) in enumerate(ranking_mediano.iterrows(), 1):
    print(f"  {idx}. {traduccion_dias[dia_eng]:<12} £{row['ticket_mediano']:.2f}")

print("\nPor VOLUMEN DE TRANSACCIONES (mayor a menor):")
ranking_trans = resumen_dias.sort_values('transacciones', ascending=False)
for idx, (dia_eng, row) in enumerate(ranking_trans.iterrows(), 1):
    print(f"  {idx}. {traduccion_dias[dia_eng]:<12} {row['transacciones']:>10,.0f} transacciones")

# 9. Conclusiones de negocio
print("\n" + "="*80)
print("  CONCLUSIONES DE NEGOCIO")
print("="*80)

coincide_texto = "SÍ coinciden" if dia_max_promedio == dia_max_mediano else "NO coinciden"

print(f"""
1. RESPUESTA A LA PREGUNTA CENTRAL:
   • Día con MAYOR ticket promedio: {traduccion_dias[dia_max_promedio]} (£{resumen_dias.loc[dia_max_promedio, 'ticket_promedio']:.2f})
   • Día con MAYOR ticket mediano : {traduccion_dias[dia_max_mediano]} (£{resumen_dias.loc[dia_max_mediano, 'ticket_mediano']:.2f})
   • ¿Coinciden? {coincide_texto.upper()}
   
   {f'→ La coincidencia indica distribuciones similares en {traduccion_dias[dia_max_promedio]}' if dia_max_promedio == dia_max_mediano else
    f'→ La NO coincidencia indica que {traduccion_dias[dia_max_promedio]} tiene más outliers de alto valor'}

2. PATRONES SEMANALES:
   • Variabilidad entre días: CV = {variabilidad_cv:.1f}% → {interpretacion}
   • {'Comportamiento homogéneo durante la semana' if variabilidad_cv < 10 else 'Diferencias claras según día de semana'}
   • Ratio Media/Mediana: todas >1 → asimetría positiva consistente

3. DIFERENCIAS TICKET MÁXIMO vs MÍNIMO:
   • Diferencia absoluta: £{mayor['ticket_promedio']-menor['ticket_promedio']:.2f}
   • Diferencia relativa: {dif_pct_promedio:+.1f}%
   • {'Diferencia significativa que requiere análisis' if abs(dif_pct_promedio) > 15 else 'Diferencia moderada entre días'}

4. VOLUMEN vs VALOR:
   • Día con más transacciones: {traduccion_dias[ranking_trans.index[0]]} ({ranking_trans.iloc[0]['transacciones']:,.0f})
   • Día con mayor ticket: {traduccion_dias[dia_max_promedio]} (£{resumen_dias.loc[dia_max_promedio, 'ticket_promedio']:.2f})
   • {'NO coinciden' if ranking_trans.index[0] != dia_max_promedio else 'SÍ coinciden'} → {'más volumen no implica mayor ticket' if ranking_trans.index[0] != dia_max_promedio else 'alto volumen con alto ticket'}

5. IMPLICACIONES ESTRATÉGICAS:

   a) STAFFING Y RECURSOS:
      • Días de alto volumen: {traduccion_dias[ranking_trans.index[0]]} requiere más personal
      • Días de alto ticket: {traduccion_dias[dia_max_promedio]} pueden requerir atención especializada
      • Distribuir recursos según volumen Y valor de transacciones

   b) PROMOCIONES Y MARKETING:
      • {'Evitar promociones profundas en ' + traduccion_dias[dia_max_promedio] + ' (ya tiene alto ticket)' if dif_pct_promedio > 10 else
         'Oportunidad de impulsar ventas con promociones'}
      • {'Estimular ticket en ' + traduccion_dias[dia_min_promedio] + ' (actualmente bajo)' if dif_pct_promedio > 10 else
         'Tickets equilibrados entre días'}
      • Up-selling/cross-selling más efectivo en días de bajo ticket

   c) GESTIÓN DE INVENTARIO:
      • Productos de alto valor disponibles especialmente en {traduccion_dias[dia_max_promedio]}
      • Planificar stock según patrón semanal de ticket y volumen
      • Analizar qué productos se venden cada día

   d) ANÁLISIS CUSTOMER:
      • ¿Diferentes tipos de clientes compran en diferentes días?
      • {traduccion_dias[dia_max_promedio] + ' puede atraer clientes mayoristas o corporativos' if dif_pct_promedio > 15 else
         'Mix de clientes aparentemente homogéneo'}
      • Segmentar campañas por día de semana

6. MÉTRICA RECOMENDADA:
   • Para operaciones diarias: usar MEDIANA (robusta ante outliers)
   • Para proyecciones revenue: usar MEDIA × volumen esperado
   • Para identificar oportunidades: analizar diferencia Media-Mediana
   • Monitorear ambas métricas: divergencia indica transacciones atípicas

7. PRÓXIMOS ANÁLISIS:
   • ¿Qué productos específicos elevan el ticket en {traduccion_dias[dia_max_promedio]}?
   • ¿Hay clientes específicos que compran solo ciertos días?
   • ¿La tendencia se mantiene mes a mes o es estacional?
   • Análisis horario dentro de cada día de semana
   • Correlación con eventos externos (pagos de salarios, etc.)

8. VALIDACIÓN ESTADÍSTICA:
   • Todas las distribuciones muestran asimetría positiva (G1 > 0)
   • Alto CV indica gran dispersión → tickets muy heterogéneos
   • La mediana es más estable que la media entre días
   • {'Diferencias estadísticamente relevantes entre días' if abs(dif_pct_promedio) > 15 else 'Diferencias moderadas entre días'}
""")

print("="*80)

  TICKET PROMEDIO POR DÍA DE SEMANA

Día              Trans.   Revenue Total  Ticket Prom.   Ticket Med.
────────────────────────────────────────────────────────────────────────────────
Lunes            64,893 £  1,367,146.41 £       21.07 £       11.90
Martes           66,473 £  1,700,634.63 £       25.58 £       12.70
Miércoles        68,885 £  1,588,336.17 £       23.06 £       13.16
Jueves           80,035 £  1,976,859.07 £       24.70 £       14.75
Viernes          54,825 £  1,485,917.40 £       27.10 £       14.85
Domingo          62,773 £    792,514.22 £       12.63 £        6.36

  ANÁLISIS COMPARATIVO

Día con MAYOR ticket promedio: Viernes (£27.10)
Día con MAYOR ticket mediano : Viernes (£14.85)

✓ SÍ COINCIDEN: Viernes tiene el mayor ticket tanto en promedio como en mediana

Día con MENOR ticket promedio: Domingo (£12.63)
Día con MENOR ticket mediano : Domingo (£6.36)

  ESTADÍSTICAS COMPLETAS POR DÍA

Día            Desvío (£)   CV (%)       G1   Ratio M/Me
────────────────

**Ejercicio 5.** Usando la regla de Tukey, detectá outliers en `Revenue` para el mercado UK. ¿Qué porcentaje de las transacciones son outliers? ¿Qué impacto tienen en la media?

In [20]:
# Solución Ejercicio 5

# 1. Filtrar datos del Reino Unido
df_uk = df[df['Country'] == 'United Kingdom'].copy()
revenue_uk = df_uk['Revenue']

print("="*85)
print("  DETECCIÓN DE OUTLIERS - REGLA DE TUKEY")
print("  Revenue del mercado UK")
print("="*85)

print(f"\nTotal de transacciones UK: {len(revenue_uk):,}")
print(f"Revenue total UK         : £{revenue_uk.sum():,.2f}")

# 2. Calcular estadísticas básicas
print("\n" + "─"*85)
print("  ESTADÍSTICAS DESCRIPTIVAS")
print("─"*85)

print(f"\nMedidas de posición:")
print(f"  Media    : £{revenue_uk.mean():,.2f}")
print(f"  Mediana  : £{revenue_uk.median():,.2f}")
print(f"  Q1 (P25) : £{revenue_uk.quantile(0.25):,.2f}")
print(f"  Q3 (P75) : £{revenue_uk.quantile(0.75):,.2f}")

print(f"\nMedidas de dispersión:")
print(f"  Mín      : £{revenue_uk.min():,.2f}")
print(f"  Máx      : £{revenue_uk.max():,.2f}")
print(f"  Rango    : £{revenue_uk.max() - revenue_uk.min():,.2f}")
print(f"  RIQ      : £{revenue_uk.quantile(0.75) - revenue_uk.quantile(0.25):,.2f}")
print(f"  Desvío   : £{revenue_uk.std():,.2f}")
print(f"  CV       : {revenue_uk.std()/revenue_uk.mean()*100:.1f}%")

print(f"\nAsimetría:")
print(f"  Fisher G1: {revenue_uk.skew():.4f}")

# 3. Aplicar regla de Tukey
Q1 = revenue_uk.quantile(0.25)
Q3 = revenue_uk.quantile(0.75)
RIQ = Q3 - Q1

# Límites de Tukey
limite_inferior = Q1 - 1.5 * RIQ
limite_superior = Q3 + 1.5 * RIQ

print("\n" + "="*85)
print("  REGLA DE TUKEY")
print("="*85)

print(f"\n  Q1 (percentil 25)           : £{Q1:,.2f}")
print(f"  Q3 (percentil 75)           : £{Q3:,.2f}")
print(f"  RIQ (Q3 - Q1)               : £{RIQ:,.2f}")

print(f"\n  Límite inferior (Q1 - 1.5×RIQ) : £{limite_inferior:,.2f}")
print(f"  Límite superior (Q3 + 1.5×RIQ) : £{limite_superior:,.2f}")

# 4. Identificar outliers
outliers_inferiores = revenue_uk[revenue_uk < limite_inferior]
outliers_superiores = revenue_uk[revenue_uk > limite_superior]
outliers_totales = revenue_uk[(revenue_uk < limite_inferior) | (revenue_uk > limite_superior)]

# Datos sin outliers
revenue_sin_outliers = revenue_uk[(revenue_uk >= limite_inferior) & (revenue_uk <= limite_superior)]

print("\n" + "="*85)
print("  DETECCIÓN DE OUTLIERS")
print("="*85)

n_total = len(revenue_uk)
n_outliers_inf = len(outliers_inferiores)
n_outliers_sup = len(outliers_superiores)
n_outliers = len(outliers_totales)
n_normales = len(revenue_sin_outliers)

pct_outliers_inf = (n_outliers_inf / n_total * 100)
pct_outliers_sup = (n_outliers_sup / n_total * 100)
pct_outliers = (n_outliers / n_total * 100)
pct_normales = (n_normales / n_total * 100)

print(f"\n{'Categoría':<30} {'Cantidad':>12} {'Porcentaje':>12}")
print("─"*85)
print(f"{'Outliers inferiores':<30} {n_outliers_inf:>12,} {pct_outliers_inf:>11.2f}%")
print(f"{'Outliers superiores':<30} {n_outliers_sup:>12,} {pct_outliers_sup:>11.2f}%")
print(f"{'TOTAL OUTLIERS':<30} {n_outliers:>12,} {pct_outliers:>11.2f}%")
print(f"{'Transacciones normales':<30} {n_normales:>12,} {pct_normales:>11.2f}%")
print("─"*85)
print(f"{'TOTAL':<30} {n_total:>12,} {100.0:>11.2f}%")

# 5. Análisis de los outliers superiores
print("\n" + "="*85)
print("  ANÁLISIS DE OUTLIERS SUPERIORES (los más impactantes)")
print("="*85)

if n_outliers_sup > 0:
    print(f"\nEstadísticas de outliers superiores:")
    print(f"  Cantidad      : {n_outliers_sup:,}")
    print(f"  Media         : £{outliers_superiores.mean():,.2f}")
    print(f"  Mediana       : £{outliers_superiores.median():,.2f}")
    print(f"  Mínimo        : £{outliers_superiores.min():,.2f}")
    print(f"  Máximo        : £{outliers_superiores.max():,.2f}")
    print(f"  Revenue total : £{outliers_superiores.sum():,.2f}")
    print(f"  % del revenue total: {(outliers_superiores.sum()/revenue_uk.sum()*100):.2f}%")
    
    # Top 10 outliers superiores
    print("\n  Top 10 OUTLIERS SUPERIORES (transacciones de mayor revenue):")
    print(f"  {'Rank':<6} {'Revenue (£)':>15} {'% del total':>12}")
    print("  " + "─"*35)
    
    top_outliers = outliers_superiores.sort_values(ascending=False).head(10)
    for idx, (trans_idx, valor) in enumerate(top_outliers.items(), 1):
        pct_individual = (valor / revenue_uk.sum() * 100)
        print(f"  {idx:<6} £{valor:>14,.2f} {pct_individual:>11.4f}%")
    
    # Detalles de las transacciones outlier
    print("\n  Detalle de top 5 transacciones outlier:")
    df_outliers = df_uk[df_uk['Revenue'].isin(top_outliers.head(5))]
    for idx, (_, row) in enumerate(df_outliers.head(5).iterrows(), 1):
        print(f"\n  {idx}. Revenue: £{row['Revenue']:,.2f}")
        print(f"     Producto : {row['Description']}")
        print(f"     Cantidad : {row['Quantity']:,.0f} unidades")
        print(f"     Precio   : £{row['UnitPrice']:.2f}")
        print(f"     Fecha    : {row['InvoiceDate'].date()}")
        print(f"     Cliente  : {int(row['CustomerID'])}")
else:
    print("\n  No se detectaron outliers inferiores.")

# 6. Comparación: Con outliers vs Sin outliers
print("\n" + "="*85)
print("  IMPACTO DE LOS OUTLIERS EN LAS ESTADÍSTICAS")
print("="*85)

# Calcular todas las métricas
metricas_comparacion = {
    'Con outliers': {
        'n': len(revenue_uk),
        'media': revenue_uk.mean(),
        'mediana': revenue_uk.median(),
        'desvio': revenue_uk.std(),
        'cv': revenue_uk.std()/revenue_uk.mean()*100,
        'min': revenue_uk.min(),
        'max': revenue_uk.max(),
        'Q1': revenue_uk.quantile(0.25),
        'Q3': revenue_uk.quantile(0.75),
        'revenue_total': revenue_uk.sum(),
        'asimetria': revenue_uk.skew()
    },
    'Sin outliers': {
        'n': len(revenue_sin_outliers),
        'media': revenue_sin_outliers.mean(),
        'mediana': revenue_sin_outliers.median(),
        'desvio': revenue_sin_outliers.std(),
        'cv': revenue_sin_outliers.std()/revenue_sin_outliers.mean()*100,
        'min': revenue_sin_outliers.min(),
        'max': revenue_sin_outliers.max(),
        'Q1': revenue_sin_outliers.quantile(0.25),
        'Q3': revenue_sin_outliers.quantile(0.75),
        'revenue_total': revenue_sin_outliers.sum(),
        'asimetria': revenue_sin_outliers.skew()
    }
}

print(f"\n{'Métrica':<20} {'Con outliers':>18} {'Sin outliers':>18} {'Cambio (%)':>15}")
print("─"*85)

for metrica in ['n', 'media', 'mediana', 'desvio', 'cv', 'min', 'max', 'Q1', 'Q3', 'asimetria']:
    val_con = metricas_comparacion['Con outliers'][metrica]
    val_sin = metricas_comparacion['Sin outliers'][metrica]
    
    if metrica == 'n':
        cambio_pct = ((val_sin - val_con) / val_con * 100)
        print(f"{metrica.upper():<20} {val_con:>18,.0f} {val_sin:>18,.0f} {cambio_pct:>+14.2f}%")
    elif metrica in ['media', 'mediana', 'desvio', 'min', 'max', 'Q1', 'Q3']:
        cambio_pct = ((val_sin - val_con) / val_con * 100)
        print(f"{metrica.capitalize():<20} £{val_con:>17,.2f} £{val_sin:>17,.2f} {cambio_pct:>+14.2f}%")
    elif metrica == 'cv':
        cambio_abs = val_sin - val_con
        print(f"{metrica.upper():<20} {val_con:>17.1f}% {val_sin:>17.1f}% {cambio_abs:>+14.1f} pp")
    elif metrica == 'asimetria':
        cambio_abs = val_sin - val_con
        print(f"Asimetría (G1)       {val_con:>18.4f} {val_sin:>18.4f} {cambio_abs:>+14.4f}")

# Revenue total
val_con = metricas_comparacion['Con outliers']['revenue_total']
val_sin = metricas_comparacion['Sin outliers']['revenue_total']
cambio_pct = ((val_sin - val_con) / val_con * 100)
print("─"*85)
print(f"{'Revenue TOTAL':<20} £{val_con:>17,.2f} £{val_sin:>17,.2f} {cambio_pct:>+14.2f}%")

# 7. Impacto específico en la media
print("\n" + "="*85)
print("  IMPACTO ESPECÍFICO EN LA MEDIA")
print("="*85)

media_con = metricas_comparacion['Con outliers']['media']
media_sin = metricas_comparacion['Sin outliers']['media']
diferencia_abs = media_con - media_sin
diferencia_pct = ((media_con - media_sin) / media_sin * 100)

print(f"\n  Media CON outliers     : £{media_con:,.2f}")
print(f"  Media SIN outliers     : £{media_sin:,.2f}")
print(f"  Diferencia absoluta    : £{diferencia_abs:,.2f}")
print(f"  Diferencia porcentual  : {diferencia_pct:+.2f}%")

print(f"\n  Interpretación:")
if diferencia_pct > 10:
    interpretacion = "ALTO - Los outliers inflan significativamente la media"
elif diferencia_pct > 5:
    interpretacion = "MODERADO - Los outliers tienen un impacto notable"
else:
    interpretacion = "BAJO - Los outliers tienen un impacto limitado"

print(f"  → Impacto: {interpretacion}")
print(f"  → Los outliers ({pct_outliers:.1f}% de transacciones) elevan la media en {diferencia_pct:.2f}%")
print(f"  → Esto equivale a £{diferencia_abs:.2f} por transacción en promedio")

# 8. Comparación mediana vs media
print("\n" + "="*85)
print("  ROBUSTEZ DE LA MEDIANA")
print("="*85)

mediana_con = metricas_comparacion['Con outliers']['mediana']
mediana_sin = metricas_comparacion['Sin outliers']['mediana']
cambio_mediana_pct = ((mediana_sin - mediana_con) / mediana_con * 100)

print(f"\n  Mediana CON outliers   : £{mediana_con:,.2f}")
print(f"  Mediana SIN outliers   : £{mediana_sin:,.2f}")
print(f"  Cambio en mediana      : {cambio_mediana_pct:+.2f}%")

print(f"\n  Media CON outliers     : £{media_con:,.2f}")
print(f"  Media SIN outliers     : £{media_sin:,.2f}")
print(f"  Cambio en media        : {diferencia_pct:+.2f}%")

print(f"\n  💡 La mediana cambia solo {abs(cambio_mediana_pct):.2f}% vs {abs(diferencia_pct):.2f}% de cambio en la media")
print(f"     → La mediana es {abs(diferencia_pct/cambio_mediana_pct if cambio_mediana_pct != 0 else float('inf')):.1f}x MÁS ROBUSTA ante outliers")

# 9. Distribución de outliers por rangos
print("\n" + "="*85)
print("  DISTRIBUCIÓN DE OUTLIERS SUPERIORES POR RANGOS")
print("="*85)

if n_outliers_sup > 0:
    # Definir rangos
    rangos_outliers = [
        (limite_superior, 1000),
        (1000, 2000),
        (2000, 5000),
        (5000, 10000),
        (10000, 20000),
        (20000, float('inf'))
    ]
    
    print(f"\n{'Rango (£)':<25} {'Cantidad':>12} {'% Outliers':>12} {'Revenue total':>18}")
    print("─"*85)
    
    for min_val, max_val in rangos_outliers:
        if max_val == float('inf'):
            rango_str = f"≥ £{min_val:,.0f}"
            count = len(outliers_superiores[outliers_superiores >= min_val])
            rev_rango = outliers_superiores[outliers_superiores >= min_val].sum()
        else:
            rango_str = f"£{min_val:,.0f} - £{max_val:,.0f}"
            count = len(outliers_superiores[(outliers_superiores >= min_val) & (outliers_superiores < max_val)])
            rev_rango = outliers_superiores[(outliers_superiores >= min_val) & (outliers_superiores < max_val)].sum()
        
        pct_count = (count / n_outliers_sup * 100) if n_outliers_sup > 0 else 0
        
        if count > 0:
            print(f"{rango_str:<25} {count:>12,} {pct_count:>11.1f}% £{rev_rango:>17,.2f}")

# 10. Conclusiones de negocio
print("\n" + "="*85)
print("  CONCLUSIONES DE NEGOCIO")
print("="*85)

print(f"""
1. RESPUESTA A LAS PREGUNTAS CLAVE:

   a) ¿Qué porcentaje son outliers?
      → {pct_outliers:.2f}% de las transacciones UK son outliers según Tukey
      → {n_outliers:,} de {n_total:,} transacciones
      → Outliers superiores: {pct_outliers_sup:.2f}% (los más impactantes)
      → Outliers inferiores: {pct_outliers_inf:.2f}% (casi inexistentes)

   b) ¿Qué impacto tienen en la media?
      → Los outliers ELEVAN la media en {diferencia_pct:+.2f}%
      → Diferencia absoluta: £{diferencia_abs:.2f} por transacción
      → {'Impacto SIGNIFICATIVO' if diferencia_pct > 10 else 'Impacto MODERADO' if diferencia_pct > 5 else 'Impacto LIMITADO'}
      → Sin outliers, el ticket promedio sería £{media_sin:.2f} (vs £{media_con:.2f} actual)

2. CARACTERIZACIÓN DE OUTLIERS:

   • Concentración: {pct_outliers:.1f}% de transacciones generan {(outliers_superiores.sum()/revenue_uk.sum()*100):.1f}% del revenue
   • {'Alta concentración del valor en pocas transacciones' if (outliers_superiores.sum()/revenue_uk.sum()*100) > 20 else 'Concentración moderada'}
   • Ticket promedio outlier: £{outliers_superiores.mean():,.2f}
   • Outlier máximo: £{outliers_superiores.max():,.2f}
   • {'Pocos outliers extremos dominan el impacto' if outliers_superiores.max() > outliers_superiores.median() * 5 else 'Outliers relativamente homogéneos'}

3. ROBUSTEZ DE ESTADÍSTICAS:

   {'Métrica':<20} {'Cambio al excluir outliers':<30}
   {'─'*50}
   Media            :  {diferencia_pct:+.2f}% → {'MUY sensible' if abs(diferencia_pct) > 10 else 'Sensible' if abs(diferencia_pct) > 5 else 'Poco sensible'}
   Mediana          :  {cambio_mediana_pct:+.2f}% → ROBUSTA (casi no cambia)
   Desvío estándar  :  {((metricas_comparacion['Sin outliers']['desvio'] - metricas_comparacion['Con outliers']['desvio'])/metricas_comparacion['Con outliers']['desvio']*100):+.2f}% → {'Muy sensible' if abs((metricas_comparacion['Sin outliers']['desvio'] - metricas_comparacion['Con outliers']['desvio'])/metricas_comparacion['Con outliers']['desvio']*100) > 20 else 'Sensible'}
   CV               :  {(metricas_comparacion['Sin outliers']['cv'] - metricas_comparacion['Con outliers']['cv']):+.1f} pp → {'Reducción significativa de dispersión' if (metricas_comparacion['Con outliers']['cv'] - metricas_comparacion['Sin outliers']['cv']) > 10 else 'Reducción moderada'}
   Asimetría (G1)   :  {(metricas_comparacion['Sin outliers']['asimetria'] - metricas_comparacion['Con outliers']['asimetria']):+.4f} → {'Distribución más simétrica sin outliers' if abs(metricas_comparacion['Sin outliers']['asimetria']) < abs(metricas_comparacion['Con outliers']['asimetria']) else 'Aún asimétrica'}

4. INTERPRETACIÓN DE OUTLIERS:

   {'Posibles causas de outliers superiores:'}
   • Pedidos mayoristas de gran volumen
   • Clientes corporativos con órdenes consolidadas
   • Eventos especiales (navidad, promociones)
   • Productos de lujo o alto valor unitario
   • Errores de carga de datos (verificar outliers extremos)

   {'¿Son outliers "legítimos" o errores?'}
   • La mayoría parecen {'legítimos (distribución Pareto típica en e-commerce)' if pct_outliers < 5 else 'requerir revisión (proporción alta de outliers)'}
   • Revenue máximo: £{outliers_superiores.max():,.2f} {'→ verificar si es error de carga' if outliers_superiores.max() > 100000 else '→ dentro de rango esperable'}
   • {'Revisar manualmente los top 10 outliers más extremos' if outliers_superiores.max() > 50000 else 'Outliers en rango razonable'}

5. IMPLICACIONES PARA REPORTES:

   a) KPIs y Métricas:
      • NO usar media como único indicador de ticket promedio
      • USAR mediana para ticket "típico": £{mediana_con:,.2f}
      • Reportar percentiles (P25, P50, P75, P90) para visión completa
      • Separar análisis de outliers vs transacciones normales

   b) Forecasting y Proyecciones:
      • Para proyección de revenue total: usar media (incluye outliers)
      • Para planificación operativa: usar mediana (más representativa)
      • Modelar outliers por separado (clientes corporativos)
      • No asumir distribución normal (usar percentiles)

   c) Segmentación de Clientes:
      • Identificar clientes que generan outliers
      • Estrategia diferenciada para "grandes cuentas"
      • Programas de retención específicos para top clientes
      • Análisis de riesgo: alta dependencia de pocos clientes

6. DECISIONES SOBRE OUTLIERS:

   {'¿Excluir outliers del análisis?'}
   • Para análisis descriptivo general: MANTENER (reflejan realidad del negocio)
   • Para análisis de cliente "típico": EXCLUIR o analizar por separado
   • Para modelos predictivos: depende del objetivo
   • Para presupuestos: MANTENER (son revenue real)

   {'¿Cuándo investigar outliers?'}
   • Outliers > £{outliers_superiores.quantile(0.99) if len(outliers_superiores) > 0 else 0:,.0f} → revisar individualmente
   • Cambios súbitos en proporción de outliers → alerta de calidad de datos
   • Outliers negativos o muy bajos → posibles errores de carga

7. ACCIONES RECOMENDADAS:

   • Investigar top 10-20 transacciones más altas (posibles errores)
   • Analizar perfil de clientes que generan outliers
   • Implementar alertas para transacciones > £{limite_superior:,.0f}
   • Reportar siempre mediana junto con media
   • Crear dashboard con percentiles (P10, P25, P50, P75, P90, P95, P99)
   • Segmentar análisis: transacciones normales vs outliers
   • Validar que outliers extremos sean transacciones reales

8. REGLA DE TUKEY EN CONTEXTO:

   • Límite superior: £{limite_superior:,.2f} ({pct_outliers_sup:.2f}% exceden este límite)
   • {'Criterio razonable' if pct_outliers_sup < 5 else 'Criterio quizás muy estricto' if pct_outliers_sup < 10 else 'Criterio muy estricto'} para este dataset
   • {'Considerar métodos alternativos (percentiles, z-score) para comparar' if pct_outliers_sup > 10 else 'Método adecuado para detectar valores atípicos'}
   • RIQ = £{RIQ:,.2f} refleja alta variabilidad natural del negocio
""")

print("="*85)

  DETECCIÓN DE OUTLIERS - REGLA DE TUKEY
  Revenue del mercado UK

Total de transacciones UK: 354,321
Revenue total UK         : £7,308,391.55

─────────────────────────────────────────────────────────────────────────────────────
  ESTADÍSTICAS DESCRIPTIVAS
─────────────────────────────────────────────────────────────────────────────────────

Medidas de posición:
  Media    : £20.63
  Mediana  : £10.20
  Q1 (P25) : £4.16
  Q3 (P75) : £17.70

Medidas de dispersión:
  Mín      : £0.00
  Máx      : £168,469.60
  Rango    : £168,469.60
  RIQ      : £13.54
  Desvío   : £326.04
  CV       : 1580.7%

Asimetría:
  Fisher G1: 431.7927

  REGLA DE TUKEY

  Q1 (percentil 25)           : £4.16
  Q3 (percentil 75)           : £17.70
  RIQ (Q3 - Q1)               : £13.54

  Límite inferior (Q1 - 1.5×RIQ) : £-16.15
  Límite superior (Q3 + 1.5×RIQ) : £38.01

  DETECCIÓN DE OUTLIERS

Categoría                          Cantidad   Porcentaje
──────────────────────────────────────────────────────────────